In [1]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve
from scipy.ndimage import gaussian_filter1d
from astropy.stats import sigma_clip
from scipy.optimize import curve_fit
import batman

# Configurações globais de plotagem
for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 10

print("="*60)
print("INICIANDO PIPELINE: AU MIC (Planetas b e c)")
print("="*60)

# ============================================================
# PASSO 1 e 2: FUNÇÕES DO MODELO ROTACIONAL (Limpeza da Estrela)
# ============================================================
def gerar_modelo_rotacional_cadencia(t, f, mask_good_flares, cadencia_s=120, janela_horas=10, sigma_clip_val=1.5):
    pontos_por_hora = 3600 / cadencia_s
    janela_pontos = int(janela_horas * pontos_por_hora)
    sigma_pontos = janela_pontos / 8
    
    quebras = list(np.where(np.diff(t) > (20 / (24 * 60)))[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]
    
    modelo_final = np.zeros_like(f)
    
    for i0, i1 in zip(seg_inicios, seg_fins):
        t_seg, f_seg, mask_seg = t[i0:i1], f[i0:i1], mask_good_flares[i0:i1]
        if len(t_seg) < janela_pontos / 4:
            modelo_final[i0:i1] = np.nanmedian(f_seg)
            continue
            
        f_limpo = np.copy(f_seg)
        for _ in range(10):
            temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
            clipped = sigma_clip(f_seg - temp_smooth, sigma_lower=10.0, sigma_upper=sigma_clip_val, maxiters=1, cenfunc='median', stdfunc='mad_std')            
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            f_limpo[~mask_seg] = temp_smooth[~mask_seg] 
            
        modelo_final[i0:i1] = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
    return modelo_final

def ajustar_trecho_especifico(t, f, mask_good_flares, t_inicio, t_fim, cadencia_s=120, janela_horas=5, sigma_upper=2.0, sigma_lower=2.0, iteracoes=6):
    buffer_dias = (janela_horas * 1.5) / 24.0 
    idx_calc = np.where((t >= t_inicio - buffer_dias) & (t <= t_fim + buffer_dias))[0]
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    if len(idx_alvo) == 0: return idx_alvo, np.array([])

    t_calc, f_calc, mask_calc = t[idx_calc], f[idx_calc], mask_good_flares[idx_calc]
    sigma_pontos = (janela_horas * (3600 / cadencia_s)) / 8
    f_limpo = np.copy(f_calc)
    
    for _ in range(iteracoes):
        temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
        clipped = sigma_clip(f_calc - temp_smooth, sigma_lower=sigma_lower, sigma_upper=sigma_upper, maxiters=1, cenfunc='median', stdfunc='mad_std')
        mascara_combinada = (~clipped.mask) & mask_calc
        t_bons, f_bons = t_calc[mascara_combinada], f_limpo[mascara_combinada]
        if len(t_bons) > 2: f_limpo = np.interp(t_calc, t_bons, f_bons)
        else: f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            
    modelo_calc = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
    return idx_alvo, modelo_calc[np.where(idx_calc == idx_alvo[0])[0][0] : np.where(idx_calc == idx_alvo[-1])[0][0] + 1]

def ajustar_trecho_linear(t, f, mask_good_flares, t_inicio, t_fim, sigma_upper=2.0, sigma_lower=3.0, offset_y=0.0, tilt=0.0):
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    if len(idx_alvo) == 0: return idx_alvo, np.array([])

    t_alvo, f_alvo = t[idx_alvo], f[idx_alvo]
    t_limpo, f_limpo = t_alvo[mask_good_flares[idx_alvo]], f_alvo[mask_good_flares[idx_alvo]]
    clipped = sigma_clip(f_limpo, sigma_lower=sigma_lower, sigma_upper=sigma_upper, maxiters=2)
    bons = ~clipped.mask

    if np.sum(bons) > 2:
        coefs = np.polyfit(t_limpo[bons], f_limpo[bons], deg=1)
        t_centro = np.mean(t_alvo)
        modelo_linear = (coefs[0] + tilt) * (t_alvo - t_centro) + np.polyval(coefs, t_centro) + offset_y
    else:
        modelo_linear = (np.ones_like(t_alvo) * np.nanmedian(f_alvo)) + offset_y
    return idx_alvo, modelo_linear

def costurar_bordas(t, modelo, bordas, tamanho_janela=25, sigma_gauss=0):
    modelo_costurado = np.copy(modelo).astype(float)
    for borda_idx in sorted(bordas):
        inicio, fim = max(0, borda_idx - tamanho_janela), min(len(t) - 1, borda_idx + tamanho_janela)
        if (fim - inicio) < 10: continue
        meia_zona = (fim - inicio) // 4
        fim_esq, ini_dir = max(inicio + 2, borda_idx - meia_zona), min(fim - 2, borda_idx + meia_zona)
        idx_esq, idx_dir = np.arange(inicio, fim_esq), np.arange(ini_dir, fim + 1)
        if len(idx_esq) < 5 or len(idx_dir) < 5: continue
            
        t_centro = t[borda_idx]
        poly_esq = np.polyfit(t[idx_esq] - t_centro, modelo_costurado[idx_esq], deg=2)
        poly_dir = np.polyfit(t[idx_dir] - t_centro, modelo_costurado[idx_dir], deg=2)
        
        zona_miolo = np.arange(fim_esq, ini_dir + 1)
        t_miolo = t[zona_miolo] - t_centro
        x_norm = np.clip((t[zona_miolo] - t[fim_esq]) / (t[ini_dir] - t[fim_esq] + 1e-10), 0, 1)
        peso = 3 * x_norm**2 - 2 * x_norm**3 
        modelo_costurado[zona_miolo] = (1 - peso) * np.polyval(poly_esq, t_miolo) + peso * np.polyval(poly_dir, t_miolo)
        
    if sigma_gauss > 0:
        quebras = list(np.where(np.diff(t) > np.median(np.diff(t)) * 5)[0] + 1)
        for i0, i1 in zip([0] + quebras, quebras + [len(t)]):
            if len(modelo_costurado[i0:i1]) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(modelo_costurado[i0:i1], sigma=sigma_gauss)
    return modelo_costurado

def ajustar_trecho_polinomial(t, f, mask_good_flares, t_inicio, t_fim, grau, sigma_upper=2.0, sigma_lower=2.0):
    """
    Ajusta um polinômio local para preservar trânsitos, isolando a região do trânsito 
    e ajustando a linha de base aos pontos fora dele usando sigma clipping.
    """
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    
    # Se a região não estiver nos dados (ex: gap de observação), retorna vazio
    if len(idx_alvo) == 0: 
        return idx_alvo, np.array([])

    t_alvo = t[idx_alvo]
    f_alvo = f[idx_alvo]
    
    # Aplica a máscara inicial de flares
    t_limpo = t_alvo[mask_good_flares[idx_alvo]]
    f_limpo = f_alvo[mask_good_flares[idx_alvo]]
    
    # O sigma_clip inferior (sigma_lower) é vital aqui para ignorar o trânsito 
    # durante o ajuste do polinômio da estrela
    clipped = sigma_clip(f_limpo, sigma_lower=sigma_lower, sigma_upper=sigma_upper, maxiters=2)
    bons = ~clipped.mask

    # Precisamos de pontos suficientes para ajustar o polinômio do grau exigido
    if np.sum(bons) > grau + 1:
        # Centralizamos o tempo para evitar instabilidade numérica na matriz do polinômio
        t_centro = np.mean(t_alvo)
        coefs = np.polyfit(t_limpo[bons] - t_centro, f_limpo[bons], deg=grau)
        modelo_poly = np.polyval(coefs, t_alvo - t_centro)
    else:
        # Fallback de segurança se o clipping remover dados demais
        modelo_poly = np.ones_like(t_alvo) * np.nanmedian(f_alvo)
        
    return idx_alvo, modelo_poly
# --- EXECUÇÃO: DOWNLOAD E LIMPEZA ---
print("\nBuscando e baixando dados de AU Mic do MAST...")
search_result = search_lightcurve("AU Mic")
indices_alvo = [0, 2, 4] # Setores 1, 27 e 95

t_todos, f_todos, modelo_todos, residuo_todos = [], [], [], []

regioes_ajuste_local = [
    [3884.6271, 3884.9503, 10.0, 1.2, 3.0, 12], [3885.9927, 3886.4632, 10.0, 5, 5.0, 12], [3900.1457, 3901.0141, 10.0, 0.8, 0.8, 10],
    [1330.2440, 1330.6343, 8, 1.2, 1.0, 12],[1347.0770, 1347.662, 20, 1.2, 1.0, 10],[2041.013, 2041.4180, 10, 1.2, 1.0, 200],    [3902.9503, 3903.4152, 10.0, 0.8, 0.8, 10],
    [2057.991, 2058.425, 8, 1.2, 1.0, 12],    #[3886.9927, 3886.4709, 8.0, 5, 5.0, 12]
    
   # ,


]
## Ajustes Polinomiais EXCLUSIVOS para proteger os trânsitos (graus 3 ou 4)
regioes_ajuste_polinomial = [
    #[1330.2440, 1330.6343, 3, 2.0, 0.5],
    #[1347.0770, 1347.6620, 3, 2.0, 0.5],
    #[2041.0130, 2041.4180, 4, 2.0, 0.5],
    [2049.4540, 2049.9840, 4, 2.0, 0.5],
    [3886.9927, 3886.4709, 3, 2.0, 0.5]
    #[2057.991, 2058.425, 2, 2.0, 0.5] ,
]

regioes_ajuste_linear = [
    [3906.9180, 3907.1300, 15.5, 9.0, 0.0001, 0.0005]
]

for idx in indices_alvo:
    print(f"A processar Índice de Busca [{idx}]...")
    lc = search_result[idx].download().remove_nans().remove_outliers()
    lc = lc[lc.quality == 0].normalize().remove_nans()

    t_sec = np.ascontiguousarray(lc.time.value, dtype=np.float64)
    f_sec = np.ascontiguousarray(lc.flux, dtype=np.float64)
    
    # Máscara para evitar que o modelo siga flares
    mask_good_flares = (f_sec - gaussian_filter1d(f_sec, sigma=15)) < (3 * np.nanstd(f_sec - gaussian_filter1d(f_sec, sigma=15)))
    
    # Gera o modelo base (onde o erro dos trânsitos ocorria)
    modelo_manchas = gerar_modelo_rotacional_cadencia(t_sec, f_sec, mask_good_flares, cadencia_s=120, janela_horas=10, sigma_clip_val=1.8)
    bordas_indices = set()

    # 1. Aplica Ajustes Locais (Anomalias Gerais da Estrela)
    for ini, fim, janela_h, sig_up, sig_low, iters in regioes_ajuste_local:
        if np.any((t_sec >= ini) & (t_sec <= fim)):
            idx_alvo, mod_local = ajustar_trecho_especifico(t_sec, f_sec, mask_good_flares, ini, fim, 120, janela_h, sig_up, sig_low, iters)
            if len(idx_alvo) > 0:
                modelo_manchas[idx_alvo] = mod_local
                bordas_indices.update([idx_alvo[0], idx_alvo[-1]])

    # 2. Aplica Ajustes Polinomiais (Proteção dos Trânsitos)
    # 2. Aplica Ajustes Polinomiais (Proteção dos Trânsitos)
    for ini, fim, grau, sig_up, sig_low in regioes_ajuste_polinomial:
        if np.any((t_sec >= ini) & (t_sec <= fim)):
            idx_alvo, mod_poly = ajustar_trecho_polinomial(t_sec, f_sec, mask_good_flares, ini, fim, grau, sig_up, sig_low)
            
            if len(idx_alvo) > 30: # Garante que tem tamanho suficiente para suavizar
                # Define quantos pontos vão se misturar nas pontas (aprox 30-40 minutos em cadência de 120s)
                pontos_blend = 15 
                
                mod_antigo = np.copy(modelo_manchas[idx_alvo])
                mod_novo = np.copy(mod_poly)
                
                # Cria rampas de transição suave (de 0 a 1)
                fade_in = np.linspace(0, 1, pontos_blend)
                fade_out = np.linspace(1, 0, pontos_blend)
                
                # Mistura o modelo velho com o novo apenas nas extremidades
                mod_novo[:pontos_blend] = mod_antigo[:pontos_blend] * (1 - fade_in) + mod_novo[:pontos_blend] * fade_in
                mod_novo[-pontos_blend:] = mod_antigo[-pontos_blend:] * (1 - fade_out) + mod_novo[-pontos_blend:] * fade_out
                
                # Aplica o modelo misturado no vetor principal
                modelo_manchas[idx_alvo] = mod_novo
                
                # Como já fizemos a transição perfeita, não precisamos mais mandar o idx_alvo pras bordas_indices
    # 3. Aplica Ajustes Lineares
    for ini, fim, sig_up, sig_low, off_y, tilt_val in regioes_ajuste_linear:
        if np.any((t_sec >= ini) & (t_sec <= fim)):
            idx_alvo, mod_linear = ajustar_trecho_linear(t_sec, f_sec, mask_good_flares, ini, fim, sig_up, sig_low, off_y, tilt_val)
            if len(idx_alvo) > 0:
                modelo_manchas[idx_alvo] = mod_linear
                bordas_indices.update([idx_alvo[0], idx_alvo[-1]])

    # Costura suave das correções com o resto da curva
    if len(bordas_indices) > 0:
        modelo_manchas_suave = costurar_bordas(t_sec, modelo_manchas, sorted(list(bordas_indices)), 25, 3)
    else:
        modelo_manchas_suave = np.copy(modelo_manchas)
        quebras = list(np.where(np.diff(t_sec) > np.median(np.diff(t_sec)) * 5)[0] + 1)
        for i0, i1 in zip([0] + quebras, quebras + [len(t_sec)]):
            if i1 - i0 > 1: modelo_manchas_suave[i0:i1] = gaussian_filter1d(modelo_manchas_suave[i0:i1], sigma=3)

    # Guarda os resultados com os trânsitos preservados
    t_todos.append(t_sec)
    f_todos.append(f_sec)
    modelo_todos.append(modelo_manchas_suave)
    residuo_todos.append(f_sec / modelo_manchas_suave)

# Vetores globais unificados
t = np.concatenate(t_todos)
residual_manchas = np.concatenate(residuo_todos)
#print("Limpeza concluída! Vetor de dados preparado e trânsitos preservados.")

INICIANDO PIPELINE: AU MIC (Planetas b e c)

Buscando e baixando dados de AU Mic do MAST...
A processar Índice de Busca [0]...
A processar Índice de Busca [2]...
A processar Índice de Busca [4]...


In [2]:
# ============================================================
# VISUALIZAÇÃO: CURVA ORIGINAL vs MODELO ROTACIONAL (Setores)
# ============================================================
%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("VISUALIZAÇÃO: DADOS ORIGINAIS vs MODELO ROTACIONAL")
print("="*60)

# Nomes dos setores
setor_nomes = ['Setor 1', 'Setor 27', 'Setor 95']

# Cria figura com 3 subplots (um para cada setor)
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for i, (t_sec, f_sec, modelo_sec) in enumerate(zip(t_todos, f_todos, modelo_todos)):
    ax = axes[i]
    
    # Plot dos dados originais
    ax.plot(t_sec, f_sec, 'k.', ms=2, alpha=0.3, label='Dados Originais (Normalizados)')
    
    # Plot do modelo rotacional
    ax.plot(t_sec, modelo_sec, 'r-', lw=2, alpha=0.8, label='Modelo Rotacional (Manchas Estelares)')
    
    # Formatação
    ax.set_ylabel('Fluxo', fontsize=11, fontweight='bold')
    ax.set_title(f'{setor_nomes[i]} - AU Mic', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, linestyle=':', linewidth=0.7)
    ax.legend(loc='upper right', fontsize=10)
    ax.set_ylim([np.nanmin(f_sec) - 0.001, np.nanmax(f_sec) + 0.001])
    
    # Informações do setor
    info_text = f"Pontos: {len(t_sec)} | T_início: {t_sec[0]:.2f} | T_fim: {t_sec[-1]:.2f} BTJD"
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Último eixo com label X
axes[-1].set_xlabel('Tempo [BTJD]', fontsize=11, fontweight='bold')

plt.suptitle('AU Mic - Qualidade do Ajuste do Modelo Rotacional por Setor', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("✓ Plots dos ajustes gerados com sucesso!\n")

# ============================================================
# ESTATÍSTICAS DO AJUSTE
# ============================================================
print("="*60)
print("ESTATÍSTICAS DOS AJUSTES POR SETOR")
print("="*60)

for i, (nome, t_sec, f_sec, modelo_sec) in enumerate(zip(setor_nomes, t_todos, f_todos, modelo_todos)):
    residuos = f_sec - modelo_sec
    residuos_norm = residuos / np.nanmean(f_sec)
    
    # Cálculo do tempo de observação
    tempo_observacao_dias = t_sec[-1] - t_sec[0]
    tempo_observacao_horas = tempo_observacao_dias * 24
    tempo_observacao_minutos = tempo_observacao_horas * 60
    
    # Cadência média (em minutos)
    dt_medio = np.median(np.diff(t_sec)) * 24 * 60  # em minutos
    
    # Duty cycle (porcentagem de tempo com dados)
    n_gaps = len(np.where(np.diff(t_sec) > 0.01)[0])  # Gaps maiores que ~15 minutos
    
    print(f"\n{'='*70}")
    print(f"  {nome.upper()}")
    print(f"{'='*70}")
    print(f"  📊 INFORMAÇÕES GERAIS:")
    print(f"     • Número de pontos: {len(t_sec):,}")
    print(f"     • Número de gaps de dados: {n_gaps}")
    print(f"     • Cadência média: {dt_medio:.1f} minutos")
    print(f"\n  ⏱️  TEMPO DE OBSERVAÇÃO:")
    print(f"     • Início: {t_sec[0]:.4f} BTJD")
    print(f"     • Fim:    {t_sec[-1]:.4f} BTJD")
    print(f"     • Duração: {tempo_observacao_dias:.2f} dias ({tempo_observacao_horas:.1f} horas, {tempo_observacao_minutos:.0f} minutos)")
    print(f"\n  💫 INFORMAÇÕES DO FLUXO:")
    print(f"     • Fluxo médio: {np.nanmean(f_sec):.6f}")
    print(f"     • Fluxo máx: {np.nanmax(f_sec):.6f}")
    print(f"     • Fluxo mín: {np.nanmin(f_sec):.6f}")
    print(f"     • Amplitude (max-min): {np.nanmax(f_sec) - np.nanmin(f_sec):.6e}")
    print(f"\n  📈 ESTATÍSTICAS DO AJUSTE:")
    print(f"     • RMS dos resíduos: {np.nanstd(residuos):.6e}")
    print(f"     • RMS relativo: {np.nanstd(residuos_norm)*1e6:.1f} ppm")
    print(f"     • Resíduo máx: {np.nanmax(np.abs(residuos)):.6e}")
    print(f"     • Resíduo mín: {np.nanmin(np.abs(residuos)):.6e}")
    print(f"     • Resíduo médio: {np.nanmean(np.abs(residuos)):.6e}")

print(f"\n{'='*70}")


VISUALIZAÇÃO: DADOS ORIGINAIS vs MODELO ROTACIONAL
✓ Plots dos ajustes gerados com sucesso!

ESTATÍSTICAS DOS AJUSTES POR SETOR

  SETOR 1
  📊 INFORMAÇÕES GERAIS:
     • Número de pontos: 17,687
     • Número de gaps de dados: 41
     • Cadência média: 2.0 minutos

  ⏱️  TEMPO DE OBSERVAÇÃO:
     • Início: 1325.9445 BTJD
     • Fim:    1353.0453 BTJD
     • Duração: 27.10 dias (650.4 horas, 39025 minutos)

  💫 INFORMAÇÕES DO FLUXO:
     • Fluxo médio: 1.004857
     • Fluxo máx: 1.051561
     • Fluxo mín: 0.984843
     • Amplitude (max-min): 6.671751e-02

  📈 ESTATÍSTICAS DO AJUSTE:
     • RMS dos resíduos: 1.354310e-03
     • RMS relativo: 1347.8 ppm
     • Resíduo máx: 3.367840e-02
     • Resíduo mín: 0.000000e+00
     • Resíduo médio: 5.586799e-04

  SETOR 27
  📊 INFORMAÇÕES GERAIS:
     • Número de pontos: 16,767
     • Número de gaps de dados: 3
     • Cadência média: 2.0 minutos

  ⏱️  TEMPO DE OBSERVAÇÃO:
     • Início: 2036.2840 BTJD
     • Fim:    2060.6469 BTJD
     • Duração

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import batman

print("\n" + "="*60)
print("PASSO 3: AJUSTE FOTOMÉTRICO COM DADOS DE LITERATURA (AU Mic b)")
print("="*60)

# ============================================================
# 1. DICIONÁRIO DE TRÂNSITOS CONHECIDOS (Setor 1 e 27)
# ============================================================
# Formato: Época: {'tc': Centro Observado, 'err': Incerteza do Centro}
transitos_conhecidos_b = {
    0:  {'tc': 1330.39046, 'err': 0.00016},
    2:  {'tc': 1347.31646, 'err': 0.00016},
    84: {'tc': 2041.28238, 'err': 0.00026},
    85: {'tc': 2049.74538, 'err': 0.00026},
    86: {'tc': 2058.20838, 'err': 0.00026}
}

# Constantes do Planeta b
T0_b, T0_b_err = 1330.39051, 0.00015
P_b, P_b_err   = 8.463000, 0.000002
RP_b, A_b, INC_b = 0.0526, 19.1, 89.5
LDC = [0.13, 0.58]

def modelo_ajuste_t0(t_janela, t0_livre, per, rp, a, inc):
    p = batman.TransitParams()
    p.t0, p.per, p.rp, p.a, p.inc = t0_livre, per, rp, a, inc
    p.ecc, p.w, p.u, p.limb_dark = 0.0, 90.0, LDC, "quadratic"
    return batman.TransitModel(p, t_janela).light_curve(p)

# ============================================================
# 2. FUNÇÃO MAPEADORA INTELIGENTE (Mistura Teoria com Literatura)
# ============================================================
def mapear_transitos_hibrido(T0, T0_err, P, P_err, t_array, janela_dias=0.25, conhecidos={}):
    n_min = int(np.ceil((t_array.min() - T0) / P))
    n_max = int(np.floor((t_array.max() - T0) / P))
    
    eventos = [] 
    for n in range(n_min, n_max + 1):
        # Se for um dos antigos conhecidos, puxa da lista que você forneceu
        if n in conhecidos:
            tc_esperado = conhecidos[n]['tc']
            tc_err_esperado = conhecidos[n]['err']
            origem = "Literatura (Conhecido)"
        # Se for um trânsito novo do Setor 95, usa a fórmula Teórica
        else:
            tc_esperado = T0 + n * P
            tc_err_esperado = np.sqrt(T0_err**2 + (n * P_err)**2)
            origem = "Teórico (Previsão Setor Novo)"
            
        # Verifica se temos dados do TESS nesse momento exato
        pontos_na_janela = np.sum((t_array >= tc_esperado - janela_dias) & (t_array <= tc_esperado + janela_dias))
        
        if pontos_na_janela > 10:
            eventos.append({
                'epoch': n, 
                'tc_esperado': tc_esperado, 
                'tc_err': tc_err_esperado,
                'origem': origem
            })
    return eventos

# Mapeia onde o planeta b passou nos nossos dados
eventos_b = mapear_transitos_hibrido(T0_b, T0_b_err, P_b, P_b_err, t, 0.25, transitos_conhecidos_b)

# ============================================================
# 3. ROTINA DE AJUSTE E PLOTAGEM (Calculando O-C para novos)
# ============================================================
%matplotlib qt

n_plots = len(eventos_b)
if n_plots > 0:
    print(f"Encontrados {n_plots} trânsitos de AU Mic b com dados!\n")
    fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4.0), sharey=True)
    if n_plots == 1: axes = [axes]

    for i, ev in enumerate(eventos_b):
        epoch = ev['epoch']
        tc_esperado = ev['tc_esperado']
        origem = ev['origem']
        
        # Isola os dados dessa janela temporal
        janela = (t >= tc_esperado - 0.25) & (t <= tc_esperado + 0.25)
        t_f, fluxo_f = t[janela], residual_manchas[janela]
        
        def wrapper_ajuste(t_j, t0_fit):
            return modelo_ajuste_t0(t_j, t0_fit, P_b, RP_b, A_b, INC_b)

        try:
            # Roda o fitting para achar o centro real nos dados
            popt, pcov = curve_fit(wrapper_ajuste, t_f, fluxo_f, p0=[tc_esperado], bounds=(tc_esperado-0.08, tc_esperado+0.08))
            tc_medido, tc_err_medido = popt[0], np.sqrt(pcov[0,0])
        except:
            tc_medido, tc_err_medido = tc_esperado, ev['tc_err']
            
        fluxo_modelo_ajustado = wrapper_ajuste(t_f, tc_medido)
        
        # --- Cálculo do O-C (Para ver os atrasos das curvas novas) ---
        # Calculamos onde o centro TEÓRICO absoluto seria
        tc_puramente_teorico = T0_b + epoch * P_b
        # Diferença do medido para o teórico (em segundos)
        o_c_segundos = (tc_medido - tc_puramente_teorico) * 24 * 3600
        
        print(f"Época {epoch:03d} | Origem Base: {origem}")
        print(f"  -> T_c Esperado: {tc_esperado:.5f}")
        print(f"  -> T_c Medido  : {tc_medido:.5f} ± {tc_err_medido:.5f}")
        print(f"  -> O-C (Atraso): {o_c_segundos:+.1f} segundos\n")
        
        # Gráficos
        ax = axes[i]
        ax.plot(t_f, fluxo_f, 'k.', ms=3, alpha=0.4)
        ax.plot(t_f, fluxo_modelo_ajustado, color='crimson', lw=2.5)
        ax.axvline(tc_medido, color='crimson', ls='--', alpha=0.5)
        ax.errorbar(tc_medido, 1.002, xerr=tc_err_medido, fmt='o', color='red', ms=4)
        
        # Título diferente se for uma curva "Nova" (Setor 95)
        cor_titulo = 'blue' if 'Novo' in origem else 'black'
        ax.set_title(f"Época {epoch}\n$T_c$: {tc_medido:.4f}\nO-C: {o_c_segundos:+.0f}s", fontsize=10, color=cor_titulo)
        ax.set_xlabel('Tempo [BTJD]')
        ax.grid(alpha=0.2)
        if i == 0: ax.set_ylabel('Fluxo Residual')

    plt.suptitle('AU Mic b - Ajuste Fotométrico (Com Prioris de Literatura e Novas Detecções)', fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()


PASSO 3: AJUSTE FOTOMÉTRICO COM DADOS DE LITERATURA (AU Mic b)


In [13]:
# ============================================================
# VISUALIZAÇÃO: CURVAS ORIGINAIS DO TELESCÓPIO (SEM AJUSTE)
# ============================================================
%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt

print("\n" + "="*60)
print("DADOS ORIGINAIS: CONFORME RECEBIDO DO TELESCÓPIO")
print("="*60)

# Nomes dos setores
setor_nomes = ['Setor 1 (2018)', 'Setor 27 (2020)', 'Setor 95(2025)']

# Cria figura com 3 subplots (um para cada setor)
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for i, (t_sec, f_sec) in enumerate(zip(t_todos, f_todos)):
    ax = axes[i]
    
    # Plot APENAS dos dados originais (sem modelo)
    ax.plot(t_sec, f_sec, 'k.-', ms=2, alpha=0.5, label='Dados Originais (Normalizados)')
    
    # ============================================================
    # ADICIONA BARRA DE DIAS OBSERVADOS NA PARTE INFERIOR
    # ============================================================
    # Calcula quais dias foram observados (arredonda para dia inteiro)
    dias_unicos = np.unique(np.floor(t_sec).astype(int))
    dias_totais = int(np.ceil(t_sec[-1])) - int(np.floor(t_sec[0])) + 1
    dias_observados = len(dias_unicos)
    
    # Cria barras para cada dia observado (altura = pequena, na base do gráfico)
    y_min, y_max = ax.get_ylim()
    altura_barra = (y_max - y_min) * 0.02  # 2% da altura do gráfico
    y_barra = y_min + altura_barra * 0.5
    
    for dia in dias_unicos:
        ax.barh(y_barra, 1.0, left=dia - 0.5, height=altura_barra * 0.3, 
                color='lightblue', edgecolor='blue', alpha=0.7, linewidth=0.5)
    
    # Formatação
    ax.set_ylabel('Fluxo Normalizado', fontsize=11, fontweight='bold')
    ax.set_title(f'{setor_nomes[i]} - AU Mic', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, linestyle=':', linewidth=0.7)
    ax.legend(loc='upper right', fontsize=10)
    ax.set_ylim([y_min, y_max])
    
    # Informações do setor - AGORA COM DIAS OBSERVADOS
    info_text = f"Dias observados: {dias_observados}/{dias_totais} | Duração: {t_sec[-1] - t_sec[0]:.2f} dias"
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

# Último eixo com label X
axes[-1].set_xlabel('Tempo [BTJD]', fontsize=11, fontweight='bold')

plt.suptitle('AU Mic TESS - Cobertura Temporal dos Dados', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print("✓ Visualização dos dados originais concluída!\n")

# ============================================================
# ESTATÍSTICAS DOS DADOS BRUTOS
# ============================================================
print("="*60)
print("ESTATÍSTICAS DOS DADOS ORIGINAIS")
print("="*60)

for i, (nome, t_sec, f_sec) in enumerate(zip(setor_nomes, t_todos, f_todos)):
    
    # Cálculo do tempo de observação
    tempo_observacao_dias = t_sec[-1] - t_sec[0]
    tempo_observacao_horas = tempo_observacao_dias * 24
    tempo_observacao_minutos = tempo_observacao_horas * 60
    
    # Cadência média (em minutos)
    dt_medio = np.median(np.diff(t_sec)) * 24 * 60
    
    # Gaps
    n_gaps = len(np.where(np.diff(t_sec) > 0.01)[0])
    
    # Dias observados
    dias_unicos = np.unique(np.floor(t_sec).astype(int))
    dias_totais = int(np.ceil(t_sec[-1])) - int(np.floor(t_sec[0])) + 1
    dias_observados = len(dias_unicos)
    duty_cycle = (dias_observados / dias_totais) * 100
    
    # Variabilidade do fluxo bruto
    variabilidade = np.nanstd(f_sec)
    amplitude = np.nanmax(f_sec) - np.nanmin(f_sec)
    
    print(f"\n{'='*70}")
    print(f"  {nome.upper()}")
    print(f"{'='*70}")
    print(f"  📊 INFORMAÇÕES GERAIS:")
    print(f"     • Número de pontos: {len(t_sec):,}")
    print(f"     • Dias observados: {dias_observados}/{dias_totais} ({duty_cycle:.1f}% cobertura)")
    print(f"     • Número de gaps: {n_gaps}")
    print(f"     • Cadência média: {dt_medio:.1f} minutos")
    print(f"\n  ⏱️  INTERVALO TEMPORAL:")
    print(f"     • Início: {t_sec[0]:.4f} BTJD")
    print(f"     • Fim:    {t_sec[-1]:.4f} BTJD")
    print(f"     • Duração: {tempo_observacao_dias:.2f} dias ({tempo_observacao_horas:.1f} horas, {tempo_observacao_minutos:.0f} minutos)")
    print(f"\n  💫 FLUXO BRUTO DO TELESCÓPIO:")
    print(f"     • Fluxo médio: {np.nanmean(f_sec):.6f}")
    print(f"     • Fluxo máx: {np.nanmax(f_sec):.6f}")
    print(f"     • Fluxo mín: {np.nanmin(f_sec):.6f}")
    print(f"     • Amplitude (max-min): {amplitude:.6e}")
    print(f"     • Desvio padrão: {variabilidade:.6e}")
    print(f"     • Variabilidade relativa: {variabilidade*1e6:.1f} ppm")

print(f"\n{'='*70}")


DADOS ORIGINAIS: CONFORME RECEBIDO DO TELESCÓPIO
✓ Visualização dos dados originais concluída!

ESTATÍSTICAS DOS DADOS ORIGINAIS

  SETOR 1 (2018)
  📊 INFORMAÇÕES GERAIS:
     • Número de pontos: 17,687
     • Dias observados: 29/30 (96.7% cobertura)
     • Número de gaps: 41
     • Cadência média: 2.0 minutos

  ⏱️  INTERVALO TEMPORAL:
     • Início: 1325.9445 BTJD
     • Fim:    1353.0453 BTJD
     • Duração: 27.10 dias (650.4 horas, 39025 minutos)

  💫 FLUXO BRUTO DO TELESCÓPIO:
     • Fluxo médio: 1.004857
     • Fluxo máx: 1.051561
     • Fluxo mín: 0.984843
     • Amplitude (max-min): 6.671751e-02
     • Desvio padrão: 1.430864e-02
     • Variabilidade relativa: 14308.6 ppm

  SETOR 27 (2020)
  📊 INFORMAÇÕES GERAIS:
     • Número de pontos: 16,767
     • Dias observados: 25/26 (96.2% cobertura)
     • Número de gaps: 3
     • Cadência média: 2.0 minutos

  ⏱️  INTERVALO TEMPORAL:
     • Início: 2036.2840 BTJD
     • Fim:    2060.6469 BTJD
     • Duração: 24.36 dias (584.7 horas,

In [14]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# =====================================================================
# 1. ENTRADA DE DADOS (Medições do seu pipeline)
# =====================================================================
# Épocas correspondentes a cada trânsito observado de AU Mic b
epocas = np.array([0, 2, 84, 85, 86, 302, 304], dtype=float)

# Tc medidos (em BTJD) obtidos pelo curve_fit no seu ajuste fotométrico
tc_medidos = np.array([
    1330.38596,  # Época 000
    1347.31851,  # Época 002
    2041.28174,  # Época 084
    2049.74505,  # Época 085
    2058.17115,  # Época 086
    3886.26528,  # Época 302 (Setor 95 - Novo)
    3903.18781   # Época 304 (Setor 95 - Novo)
])

# Incertezas formais (1-sigma) de cada Tc medido
tc_erros = np.array([
    0.00063,
    0.00095,
    0.00059,
    0.00084,
    0.00136,
    0.00137,
    0.00164
])

# =====================================================================
# 2. DEFINIÇÃO DO MODELO LINEAR DE EFEMÉRIDE
# =====================================================================
def modelo_efemeride(E, T0, P):
    return T0 + E * P

# Palpites iniciais aproximados (vêm da literatura)
p0 = [1330.39051, 8.463]

# =====================================================================
# 3. AJUSTE LINEAR PONDERADO (Weighted Least Squares)
# =====================================================================
# 'sigma' recebe os erros experimentais. 'absolute_sigma=True' garante
# que as incertezas dos parâmetros sejam escaladas corretamente pelos erros físicos.
popt, pcov = curve_fit(
    modelo_efemeride, 
    epocas, 
    tc_medidos, 
    p0=p0, 
    sigma=tc_erros, 
    absolute_sigma=True
)

# Extração dos parâmetros ajustados e suas incertezas (1-sigma)
T0_novo, P_novo = popt
T0_novo_err, P_novo_err = np.sqrt(np.diag(pcov))

print("=" * 60)
# Usando formato padrão de texto e símbolos para os resultados numéricos
print("NOVA EFEMÉRIDE ESTIMADA (AU Mic b):")
print("=" * 60)
print(f"Época de Referência (T0) : {T0_novo:.6f} ± {T0_novo_err:.6f} BTJD")
print(f"Período Orbital (P)      : {P_novo:.7f} ± {P_novo_err:.7f} dias")
print(f"                         : {P_novo * 24:.5f} horas")
print("=" * 60)

# =====================================================================
# 4. CÁLCULO DOS RESÍDUOS O-C (Observado - Calculado)
# =====================================================================
# Tc calculados com base na nova efeméride recém-ajustada
tc_calculados = modelo_efemeride(epocas, T0_novo, P_novo)

# Resíduos em dias e em minutos
oc_dias = tc_medidos - tc_calculados
oc_minutos = oc_dias * 24.0 * 60.0
oc_erros_minutos = tc_erros * 24.0 * 60.0

# Print tabela de resíduos
print("\nTabela de Resíduos (O-C):")
print(f"{'Época':<8}{'O-C (dias)':<15}{'O-C (minutos)':<18}{'Erro (minutos)':<15}")
for ep, o_c, o_c_m, err_m in zip(epocas, oc_dias, oc_minutos, oc_erros_minutos):
   print(f"{int(ep):<8}{o_c:<+15.5f}{o_c_m:<+18.2f}{err_m:<15.2f}")

# Chi-quadrado reduzido (para checar a qualidade estatística do ajuste)
chi2 = np.sum(((tc_medidos - tc_calculados) / tc_erros) ** 2)
graus_liberdade = len(epocas) - len(popt)
chi2_red = chi2 / graus_liberdade
print(f"\nChi-quadrado Reduzido (X²_red): {chi2_red:.2f}")

# =====================================================================
# 5. VISUALIZAÇÃO DO DIAGRAMA O-C
# =====================================================================
plt.figure(figsize=(10, 5))

# Plotar pontos conhecidos (Setores 1 e 27) vs Novos (Setor 95)
mask_novos = epocas >= 300
plt.errorbar(epocas[~mask_novos], oc_minutos[~mask_novos], yerr=oc_erros_minutos[~mask_novos],
             fmt='o', color='black', label='Setores 1 & 27 (Anteriores)', capsize=3, ms=6)
plt.errorbar(epocas[mask_novos], oc_minutos[mask_novos], yerr=oc_erros_minutos[mask_novos],
             fmt='s', color='crimson', label='Setor 95 (Novos)', capsize=3, ms=7)

# Linha de base zero (efeméride perfeita ajustada)
plt.axhline(0, color='gray', linestyle='--', alpha=0.7)

plt.title('Diagrama O-C de AU Mic b (Nova Efeméride Linear)', fontsize=12, fontweight='bold')
plt.xlabel('Época (Ciclos)', fontsize=10)
plt.ylabel('O-C (minutos)', fontsize=10)
plt.grid(alpha=0.3, linestyle=':')
plt.legend(loc='best')
plt.tight_layout()
plt.show()


# =====================================================================
# 5. VISUALIZAÇÃO DO DIAGRAMA O-C (Eixo X = Tempo Médio do Trânsito)
# =====================================================================
plt.figure(figsize=(10, 5))

# Plotar pontos conhecidos (Setores 1 e 27) vs Novos (Setor 95)
mask_novos = epocas >= 300

# Alterado de 'epocas' para 'tc_medidos' no eixo X
plt.errorbar(tc_medidos[~mask_novos], oc_minutos[~mask_novos], yerr=oc_erros_minutos[~mask_novos],
             fmt='o', color='black', label='Setores 1 & 27 (Anteriores)', capsize=3, ms=6)

plt.errorbar(tc_medidos[mask_novos], oc_minutos[mask_novos], yerr=oc_erros_minutos[mask_novos],
             fmt='s', color='crimson', label='Setor 95 (Novos)', capsize=3, ms=7)

# Linha de base zero (efeméride perfeita ajustada)
plt.axhline(0, color='gray', linestyle='--', alpha=0.7)

plt.title('Diagrama O-C de AU Mic b (Nova Efeméride Linear)', fontsize=12, fontweight='bold')
# Rótulo do eixo X atualizado para refletir o tempo do trânsito
plt.xlabel('Tempo do Trânsito medido ($T_c$) [BTJD]', fontsize=10)
plt.ylabel('O-C (minutos)', fontsize=10)
plt.grid(alpha=0.3, linestyle=':')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

NOVA EFEMÉRIDE ESTIMADA (AU Mic b):
Época de Referência (T0) : 1330.380549 ± 0.000431 BTJD
Período Orbital (P)      : 8.4631516 ± 0.0000039 dias
                         : 203.11564 horas

Tabela de Resíduos (O-C):
Época   O-C (dias)     O-C (minutos)     Erro (minutos) 
0       +0.00541       +7.79             0.91           
2       +0.01166       +16.79            1.37           
84      -0.00355       -5.11             0.85           
85      -0.00339       -4.88             1.21           
86      -0.04044       -58.23            1.96           
302     +0.01294       +18.63            1.97           
304     +0.00917       +13.20            2.36           

Chi-quadrado Reduzido (X²_red): 256.26


In [ ]:
# =====================================================================
# 6. VISUALIZAÇÃO: CURVA DE LUZ FASEADA COM A NOVA EFEMÉRIDE
# =====================================================================
# Nota: Este código usa as matrizes 't' e 'residual_manchas' geradas 
# nos passos anteriores do seu pipeline.

# Calcula a fase para todos os pontos de dados (centralizada em 0)
fase_dias = ((t - T0_novo + 0.5 * P_novo) % P_novo) - 0.5 * P_novo
fase_horas = fase_dias * 24.0

# Define uma janela de visualização ao redor do trânsito (ex: +/- 5 horas)
janela_horas = 5.0
mask_transito = np.abs(fase_horas) < janela_horas

t_faseado = fase_horas[mask_transito]
fluxo_faseado = residual_manchas[mask_transito]

plt.figure(figsize=(10, 6))

# 1. Plota todos os dados sobrepostos (Phase-Folded)
plt.plot(t_faseado, fluxo_faseado, 'k.', ms=3, alpha=0.3, label='Dados do TESS Faseados')

# 2. Calcula e plota a média em bins (Binning) para evidenciar a forma do trânsito
bins = np.linspace(-janela_horas, janela_horas, 150)
from scipy.stats import binned_statistic
bin_means, bin_edges, _ = binned_statistic(t_faseado, fluxo_faseado, statistic='mean', bins=bins)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

plt.plot(bin_centers, bin_means, 'r-', lw=2.5, label='Fluxo Médio (Binned)')

# 3. Formatação do Gráfico
plt.axvline(0, color='blue', linestyle='--', alpha=0.7, lw=2, label='Centro Previsto ($T_c$) - Nova Efeméride')

plt.title('AU Mic b: Trânsitos Faseados pela Nova Efeméride Linear', fontsize=14, fontweight='bold')
plt.xlabel('Fase (Horas a partir do centro do trânsito)', fontsize=12)
plt.ylabel('Fluxo Residual Normalizado', fontsize=12)
plt.xlim([-janela_horas, janela_horas])
plt.ylim([np.nanmin(fluxo_faseado) - 0.002, np.nanmax(fluxo_faseado) + 0.002])
plt.grid(alpha=0.3, linestyle=':')
plt.legend(loc='lower left', fontsize=10)
plt.tight_layout()
plt.show()

NameError: name 't' is not defined

: 

In [55]:
# =====================================================================
# 7. VISUALIZAÇÃO: TODOS OS TRÂNSITOS INDIVIDUAIS (NOVA EFEMÉRIDE)
# =====================================================================
# Nota: Usa os vetores globais 't' e 'residual_manchas' do seu pipeline.

plt.figure(figsize=(16, 8))

# Prepara os subplots (2 linhas x 4 colunas, para caber os 7 trânsitos)
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
axes = axes.flatten()

# Tamanho do recorte em dias ao redor do centro (ex: +/- 3.6 horas)
janela_dias = 0.15 

for i, ep in enumerate(epocas):
    ax = axes[i]
    
    # 1. Calcula o centro exato previsto pela NOVA efeméride
    tc_previsto = T0_novo + ep * P_novo
    
    # 2. Isola os dados nesta janela temporal
    mask = (t >= tc_previsto - janela_dias) & (t <= tc_previsto + janela_dias)
    t_janela = t[mask]
    f_janela = residual_manchas[mask]
    
    # 3. Plota os dados daquele trânsito específico
    ax.plot(t_janela, f_janela, 'k.-', lw=1, ms=5, alpha=0.6, label='Dados (residual)')
    
    # 4. Plota uma linha marcando o centro da nova efeméride
    ax.axvline(tc_previsto, color='blue', ls='--', lw=2, alpha=0.7, label='Nova Efeméride')
    
    # Opcional: Adiciona um fundo sombreado leve
    ax.fill_between(t_janela, 0.99, 1.01, alpha=0.1, color='gray')
    
    # 5. Formatação do subplot
    ax.set_title(f"Época {int(ep)}\n$T_c$ = {tc_previsto:.4f}", fontsize=10, fontweight='bold')
    ax.set_xlabel("Tempo [BTJD]", fontsize=9)
    if i % 4 == 0:
        ax.set_ylabel("Fluxo Residual", fontsize=9)
        
    ax.grid(alpha=0.3)
    ax.set_xlim([tc_previsto - janela_dias, tc_previsto + janela_dias])
    ax.set_ylim([0.9925, 1.0075]) # Mantém a mesma escala visual do seu pipeline
    
    # Coloca a legenda apenas no primeiro gráfico para não poluir
    if i == 0:
        ax.legend(loc='lower left', fontsize=8)

# Remove o último subplot (índice 7), pois temos apenas 7 trânsitos (0 a 6)
fig.delaxes(axes[7])

plt.suptitle('AU Mic b - Trânsitos Observados vs. Previsão da Nova Efeméride Linear', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Valores de Tc calculados pela Nova Efeméride (BTJD):")
for ep, tc_calc in zip(epocas, tc_calculados):
    print(f"Época {int(ep):<4} -> Tc = {tc_calc:.5f}")

Valores de Tc calculados pela Nova Efeméride (BTJD):
Época 0    -> Tc = 1330.38055
Época 2    -> Tc = 1347.30685
Época 84   -> Tc = 2041.28529
Época 85   -> Tc = 2049.74844
Época 86   -> Tc = 2058.21159
Época 302  -> Tc = 3886.25234
Época 304  -> Tc = 3903.17864


In [56]:
# ============================================================
# VISUALIZAÇÃO: CURVAS ORIGINAIS DO TELESCÓPIO + NOVA EFEMÉRIDE
# ============================================================
%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt


print("\n" + "="*60)
print("DADOS ORIGINAIS + NOVA EFEMÉRIDE AU Mic b")
print("="*60)


# ============================================================
# NOVA EFEMÉRIDE AU Mic b
# ============================================================

T0_new = 1330.380549       # BTJD
P_new = 8.4631516          # dias

erro_T0 = 0.000431
erro_P = 0.0000039


print("\nNova efeméride:")
print(f"T0 = {T0_new:.6f} BTJD")
print(f"P  = {P_new:.7f} dias")
print(f"P  = {P_new*24:.5f} horas")


# ============================================================
# CALCULA TEMPOS DOS TRÂNSITOS
# ============================================================

tempo_min = np.min([np.min(t) for t in t_todos])
tempo_max = np.max([np.max(t) for t in t_todos])


# épocas que cobrem todos os dados
n_min = int(np.floor((tempo_min - T0_new)/P_new))
n_max = int(np.ceil((tempo_max - T0_new)/P_new))


epocas = np.arange(n_min, n_max+1)

transitos = T0_new + epocas*P_new


print("\n" + "="*60)
print("TRÂNSITOS PREVISTOS PELA NOVA EFEMÉRIDE")
print("="*60)

for n, t in zip(epocas, transitos):
    print(f"Época {n:4d}  -->  Trânsito = {t:.6f} BTJD")


# ============================================================
# NOMES DOS SETORES
# ============================================================

setor_nomes = [
    'Setor 1 (2018)',
    'Setor 27 (2020)',
    'Setor 95 (2025)'
]


# ============================================================
# FIGURA
# ============================================================

fig, axes = plt.subplots(
    3,1,
    figsize=(14,10),
    sharex=False
)


for i,(t_sec,f_sec) in enumerate(zip(t_todos,f_todos)):

    ax = axes[i]


    # Dados originais
    ax.plot(
        t_sec,
        f_sec,
        'k.-',
        ms=2,
        alpha=0.5,
        label='Dados Originais'
    )


    # ========================================================
    # MARCA OS TRÂNSITOS DA NOVA EFEMÉRIDE
    # ========================================================

    for j,tr in enumerate(transitos):

        if t_sec[0] <= tr <= t_sec[-1]:

            ax.axvline(
                tr,
                color='red',
                linestyle='--',
                linewidth=1.5,
                alpha=0.8,
                label='Trânsito previsto' if j==0 else None
            )

            ax.text(
                tr,
                np.nanmax(f_sec),
                f"E{epocas[j]}",
                rotation=90,
                fontsize=8,
                color='red',
                verticalalignment='top'
            )


    # ========================================================
    # BARRA DOS DIAS OBSERVADOS
    # ========================================================

    dias_unicos = np.unique(
        np.floor(t_sec).astype(int)
    )

    dias_totais = (
        int(np.ceil(t_sec[-1]))
        -
        int(np.floor(t_sec[0]))
        +
        1
    )

    dias_observados=len(dias_unicos)


    y_min,y_max=ax.get_ylim()

    altura_barra=(y_max-y_min)*0.02

    y_barra=y_min+altura_barra*0.5


    for dia in dias_unicos:

        ax.barh(
            y_barra,
            1.0,
            left=dia-0.5,
            height=altura_barra*0.3,
            color='lightblue',
            edgecolor='blue',
            alpha=0.7
        )


    # Formatação

    ax.set_ylabel(
        'Fluxo Normalizado',
        fontsize=11,
        fontweight='bold'
    )


    ax.set_title(
        f'{setor_nomes[i]} - AU Mic',
        fontsize=12,
        fontweight='bold'
    )


    ax.grid(
        alpha=0.3,
        linestyle=':',
        linewidth=0.7
    )


    ax.legend(
        fontsize=9,
        loc='upper right'
    )


    info_text = (
        f"Dias observados: {dias_observados}/{dias_totais}\n"
        f"Duração: {t_sec[-1]-t_sec[0]:.2f} dias"
    )


    ax.text(
        0.02,
        0.95,
        info_text,
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment='top',
        bbox=dict(
            boxstyle='round',
            facecolor='lightblue',
            alpha=0.5
        )
    )


axes[-1].set_xlabel(
    'Tempo [BTJD]',
    fontsize=11,
    fontweight='bold'
)



plt.suptitle(
    'AU Mic b - Cobertura Temporal TESS + Trânsitos previstos pela nova efeméride',
    fontsize=14,
    fontweight='bold'
)


plt.tight_layout()

plt.show()



# ============================================================
# ESTATÍSTICAS
# ============================================================

print("\n✓ Visualização concluída com nova efeméride!")
print("="*60)


DADOS ORIGINAIS + NOVA EFEMÉRIDE AU Mic b

Nova efeméride:
T0 = 1330.380549 BTJD
P  = 8.4631516 dias
P  = 203.11564 horas

TRÂNSITOS PREVISTOS PELA NOVA EFEMÉRIDE
Época   -1  -->  Trânsito = 1321.917397 BTJD
Época    0  -->  Trânsito = 1330.380549 BTJD
Época    1  -->  Trânsito = 1338.843701 BTJD
Época    2  -->  Trânsito = 1347.306852 BTJD
Época    3  -->  Trânsito = 1355.770004 BTJD
Época    4  -->  Trânsito = 1364.233155 BTJD
Época    5  -->  Trânsito = 1372.696307 BTJD
Época    6  -->  Trânsito = 1381.159459 BTJD
Época    7  -->  Trânsito = 1389.622610 BTJD
Época    8  -->  Trânsito = 1398.085762 BTJD
Época    9  -->  Trânsito = 1406.548913 BTJD
Época   10  -->  Trânsito = 1415.012065 BTJD
Época   11  -->  Trânsito = 1423.475217 BTJD
Época   12  -->  Trânsito = 1431.938368 BTJD
Época   13  -->  Trânsito = 1440.401520 BTJD
Época   14  -->  Trânsito = 1448.864671 BTJD
Época   15  -->  Trânsito = 1457.327823 BTJD
Época   16  -->  Trânsito = 1465.790975 BTJD
Época   17  -->  Trânsito 

In [57]:
# ============================================================
# COMPARAÇÃO EFEMÉRIDE ANTIGA VS NOVA - AU Mic b
# ============================================================

%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt


print("="*60)
print("COMPARAÇÃO DAS EFEMÉRIDES AU Mic b")
print("="*60)


# ============================================================
# EFEMÉRIDE ANTIGA
# ============================================================

T0_old = 1330.39046
erro_T0_old = 0.00016

# coloque aqui o período antigo
P_old = 8.4631516   # substituir se diferente


# ============================================================
# EFEMÉRIDE NOVA
# ============================================================

T0_new = 1330.380549
erro_T0_new = 0.000431

P_new = 8.4631516


print("\nEFEMÉRIDE ANTIGA")
print(f"T0 = {T0_old:.6f} ± {erro_T0_old:.6f}")
print(f"P  = {P_old:.7f} dias")


print("\nEFEMÉRIDE NOVA")
print(f"T0 = {T0_new:.6f} ± {erro_T0_new:.6f}")
print(f"P  = {P_new:.7f} dias")


# diferença

delta_T0 = (T0_old-T0_new)*24*60

print("\nDiferença entre T0:")
print(f"{delta_T0:.2f} minutos")


# ============================================================
# CALCULA TRÂNSITOS
# ============================================================

tempo_min = np.min([np.min(t) for t in t_todos])
tempo_max = np.max([np.max(t) for t in t_todos])


nmin_old = int(np.floor((tempo_min-T0_old)/P_old))
nmax_old = int(np.ceil((tempo_max-T0_old)/P_old))


ep_old = np.arange(nmin_old,nmax_old+1)

trans_old = T0_old + ep_old*P_old



nmin_new = int(np.floor((tempo_min-T0_new)/P_new))
nmax_new = int(np.ceil((tempo_max-T0_new)/P_new))


ep_new = np.arange(nmin_new,nmax_new+1)

trans_new = T0_new + ep_new*P_new



# ============================================================
# PLOT
# ============================================================

setor_nomes=[
    'Setor 1 (2018)',
    'Setor 27 (2020)',
    'Setor 95 (2025)'
]


fig,axes=plt.subplots(
    3,1,
    figsize=(14,10)
)


for i,(t_sec,f_sec) in enumerate(zip(t_todos,f_todos)):


    ax=axes[i]


    ax.plot(
        t_sec,
        f_sec,
        'k.',
        ms=2,
        alpha=0.5,
        label="Dados TESS"
    )


    # -----------------------------
    # ANTIGA
    # -----------------------------

    for j,tr in enumerate(trans_old):

        if t_sec[0] <= tr <= t_sec[-1]:

            ax.axvline(
                tr,
                color='blue',
                linestyle='--',
                linewidth=1.5,
                label="Efeméride antiga"
                if j==0 else None
            )


    # -----------------------------
    # NOVA
    # -----------------------------

    for j,tr in enumerate(trans_new):

        if t_sec[0] <= tr <= t_sec[-1]:

            ax.axvline(
                tr,
                color='red',
                linestyle='-',
                linewidth=1.5,
                label="Efeméride nova"
                if j==0 else None
            )


    ax.set_title(
        setor_nomes[i],
        fontsize=12,
        fontweight='bold'
    )


    ax.set_ylabel(
        "Fluxo normalizado"
    )


    ax.grid(alpha=0.3)

    ax.legend()



axes[-1].set_xlabel(
    "BTJD"
)


plt.suptitle(
    "AU Mic b - Comparação Efeméride Antiga (azul) vs Nova (vermelho)",
    fontsize=14,
    fontweight='bold'
)


plt.tight_layout()

plt.show()

COMPARAÇÃO DAS EFEMÉRIDES AU Mic b

EFEMÉRIDE ANTIGA
T0 = 1330.390460 ± 0.000160
P  = 8.4631516 dias

EFEMÉRIDE NOVA
T0 = 1330.380549 ± 0.000431
P  = 8.4631516 dias

Diferença entre T0:
14.27 minutos


In [11]:
# ============================================================
# COMPARAÇÃO EFEMÉRIDE OLD VS NOVA - AU Mic b
# COM ERROS DE T0
# ============================================================

%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt


print("\n" + "="*70)
print("COMPARAÇÃO DAS EFEMÉRIDES AU Mic b")
print("="*70)


# ============================================================
# EFEMÉRIDE OLD
# ============================================================

T0_old = 1330.39051
erro_T0_old = 0.00016

# colocar aqui o período antigo quando disponível
P_old = 8.4631516


# ============================================================
# EFEMÉRIDE NOVA
# ============================================================

T0_new = 1330.388043
erro_T0_new = 0.000530

P_new = 8.4631516


print("\nEFEMÉRIDE OLD")
print("-"*40)
print(f"T0 = {T0_old:.6f} ± {erro_T0_old:.6f} BTJD")
print(f"P  = {P_old:.7f} dias")
print(f"Erro T0 = {erro_T0_old*24*60:.2f} minutos")


print("\nEFEMÉRIDE NOVA")
print("-"*40)
print(f"T0 = {T0_new:.6f} ± {erro_T0_new:.6f} BTJD")
print(f"P  = {P_new:.7f} dias")
print(f"Erro T0 = {erro_T0_new*24*60:.2f} minutos")


# diferença entre efemérides

delta = (T0_old - T0_new)*24*60

print("\nDiferença entre T0:")
print(f"{delta:.2f} minutos")


# ============================================================
# INTERVALO DOS DADOS
# ============================================================

tempo_min = np.min([np.min(t) for t in t_todos])
tempo_max = np.max([np.max(t) for t in t_todos])


# ============================================================
# CALCULA TRÂNSITOS OLD
# ============================================================

nmin_old = int(np.floor((tempo_min-T0_old)/P_old))
nmax_old = int(np.ceil((tempo_max-T0_old)/P_old))

ep_old = np.arange(
    nmin_old,
    nmax_old+1
)

trans_old = T0_old + ep_old*P_old



# ============================================================
# CALCULA TRÂNSITOS NOVA
# ============================================================

nmin_new = int(np.floor((tempo_min-T0_new)/P_new))
nmax_new = int(np.ceil((tempo_max-T0_new)/P_new))

ep_new = np.arange(
    nmin_new,
    nmax_new+1
)

trans_new = T0_new + ep_new*P_new



# ============================================================
# MOSTRA LISTA DOS TRÂNSITOS
# ============================================================

print("\n")
print("="*70)
print("TRÂNSITOS PREVISTOS")
print("="*70)


print("\nOLD")
for e,t in zip(ep_old,trans_old):
    print(f"Época {e:4d}  {t:.6f} BTJD")


print("\nNOVA")
for e,t in zip(ep_new,trans_new):
    print(f"Época {e:4d}  {t:.6f} BTJD")



# ============================================================
# NOMES DOS SETORES
# ============================================================

setor_nomes = [
    "Setor 1 (2018)",
    "Setor 27 (2020)",
    "Setor 95 (2025)"
]



# ============================================================
# FIGURA
# ============================================================

fig, axes = plt.subplots(
    3,
    1,
    figsize=(14,10)
)



for i,(t_sec,f_sec) in enumerate(zip(t_todos,f_todos)):

    ax = axes[i]


    # dados

    ax.plot(
        t_sec,
        f_sec,
        'k.',
        markersize=2,
        alpha=0.5,
        label="Dados TESS"
    )



    # ========================================================
    # OLD
    # ========================================================

    first_old=True

    for e,tr in zip(ep_old,trans_old):

        if t_sec[0] <= tr <= t_sec[-1]:


            ax.axvline(
                tr,
                color='blue',
                linestyle='--',
                linewidth=1.8,
                label="Efeméride OLD" if first_old else None
            )


            ax.axvspan(
                tr-erro_T0_old,
                tr+erro_T0_old,
                color='blue',
                alpha=0.15,
                label="Erro OLD (1σ)" if first_old else None
            )


            ax.text(
                tr,
                np.nanmax(f_sec),
                f"E{e}",
                color="blue",
                rotation=90,
                fontsize=8
            )


            first_old=False



    # ========================================================
    # NOVA
    # ========================================================

    first_new=True

    for e,tr in zip(ep_new,trans_new):

        if t_sec[0] <= tr <= t_sec[-1]:


            ax.axvline(
                tr,
                color='red',
                linestyle='-',
                linewidth=1.8,
                label="Efeméride NOVA" if first_new else None
            )


            ax.axvspan(
                tr-erro_T0_new,
                tr+erro_T0_new,
                color='red',
                alpha=0.15,
                label="Erro NOVA (1σ)" if first_new else None
            )


            ax.text(
                tr,
                np.nanmin(f_sec),
                f"E{e}",
                color="red",
                rotation=90,
                fontsize=8
            )


            first_new=False



    # ========================================================
    # FORMATAÇÃO
    # ========================================================

    ax.set_title(
        setor_nomes[i],
        fontsize=13,
        fontweight='bold'
    )


    ax.set_ylabel(
        "Fluxo normalizado"
    )


    ax.grid(
        alpha=0.3,
        linestyle=":"
    )


    ax.legend(
        fontsize=9,
        loc="best"
    )



axes[-1].set_xlabel(
    "Tempo [BTJD]",
    fontsize=12
)



plt.suptitle(
    "AU Mic b - Comparação Efeméride OLD vs NOVA\n"
    "Azul = OLD | Vermelho = NOVA",
    fontsize=15,
    fontweight="bold"
)


plt.tight_layout()

plt.show()



print("\n✓ Comparação finalizada!")


COMPARAÇÃO DAS EFEMÉRIDES AU Mic b

EFEMÉRIDE OLD
----------------------------------------
T0 = 1330.390510 ± 0.000160 BTJD
P  = 8.4631516 dias
Erro T0 = 0.23 minutos

EFEMÉRIDE NOVA
----------------------------------------
T0 = 1330.388043 ± 0.000530 BTJD
P  = 8.4631516 dias
Erro T0 = 0.76 minutos

Diferença entre T0:
3.55 minutos


TRÂNSITOS PREVISTOS

OLD
Época   -1  1321.927358 BTJD
Época    0  1330.390510 BTJD
Época    1  1338.853662 BTJD
Época    2  1347.316813 BTJD
Época    3  1355.779965 BTJD
Época    4  1364.243116 BTJD
Época    5  1372.706268 BTJD
Época    6  1381.169420 BTJD
Época    7  1389.632571 BTJD
Época    8  1398.095723 BTJD
Época    9  1406.558874 BTJD
Época   10  1415.022026 BTJD
Época   11  1423.485178 BTJD
Época   12  1431.948329 BTJD
Época   13  1440.411481 BTJD
Época   14  1448.874632 BTJD
Época   15  1457.337784 BTJD
Época   16  1465.800936 BTJD
Época   17  1474.264087 BTJD
Época   18  1482.727239 BTJD
Época   19  1491.190390 BTJD
Época   20  1499.653542 BTJD


In [12]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import batman

print("\n" + "="*60)
print("PASSO 3: AJUSTE FOTOMÉTRICO COM DADOS DE LITERATURA (AU Mic b)")
print("COMPARAÇÃO EFEMÉRIDES OLD vs NEW")
print("="*60)

# ============================================================
# EFEMÉRIDES: OLD vs NEW
# ============================================================
class Ephemeris:
    def __init__(self, name, T0, T0_err, P, P_err=0):
        self.name = name
        self.T0 = T0
        self.T0_err = T0_err
        self.P = P
        self.P_err = P_err
    
    def __repr__(self):
        return f"{self.name}: T0={self.T0:.6f}±{self.T0_err:.6f}, P={self.P:.7f}"

eph_old = Ephemeris(
    name="OLD",
    T0=1330.39051, 
    T0_err=0.00016,
    P=8.4631516,
    P_err=0
)

eph_new = Ephemeris(
    name="NEW",
    T0=1330.380549,
    T0_err=0.000431,
    P=8.4631516,
    P_err=0
)

print(f"\n{eph_old}")
print(f"{eph_new}")
print(f"\nDiferença ΔT0 (OLD - NEW): {(eph_old.T0 - eph_new.T0)*24*3600:.3f} segundos")

# ============================================================
# 1. DICIONÁRIO DE TRÂNSITOS CONHECIDOS (Setor 1 e 27)
# ============================================================
transitos_conhecidos_b = {
    0:  {'tc': 1330.39046, 'err': 0.00016},
    2:  {'tc': 1347.31646, 'err': 0.00016},
    84: {'tc': 2041.28238, 'err': 0.00026},
    85: {'tc': 2049.74538, 'err': 0.00026},
    86: {'tc': 2058.20838, 'err': 0.00026}
}

# Constantes do Planeta b
RP_b, A_b, INC_b = 0.0526, 19.1, 89.5
LDC = [0.13, 0.58]

def modelo_ajuste_t0(t_janela, t0_livre, per, rp, a, inc):
    p = batman.TransitParams()
    p.t0, p.per, p.rp, p.a, p.inc = t0_livre, per, rp, a, inc
    p.ecc, p.w, p.u, p.limb_dark = 0.0, 90.0, LDC, "quadratic"
    return batman.TransitModel(p, t_janela).light_curve(p)

# ============================================================
# 2. FUNÇÃO MAPEADORA INTELIGENTE (Mistura Teoria com Literatura)
# ============================================================
def mapear_transitos_hibrido(ephemeris, t_array, janela_dias=0.25, conhecidos={}):
    n_min = int(np.ceil((t_array.min() - ephemeris.T0) / ephemeris.P))
    n_max = int(np.floor((t_array.max() - ephemeris.T0) / ephemeris.P))
    
    eventos = [] 
    for n in range(n_min, n_max + 1):
        if n in conhecidos:
            tc_esperado = conhecidos[n]['tc']
            tc_err_esperado = conhecidos[n]['err']
            origem = "Literatura (Conhecido)"
        else:
            tc_esperado = ephemeris.T0 + n * ephemeris.P
            tc_err_esperado = np.sqrt(ephemeris.T0_err**2 + (n * ephemeris.P_err)**2)
            origem = "Teórico (Previsão)"
            
        pontos_na_janela = np.sum((t_array >= tc_esperado - janela_dias) & (t_array <= tc_esperado + janela_dias))
        
        if pontos_na_janela > 10:
            eventos.append({
                'epoch': n, 
                'tc_esperado': tc_esperado, 
                'tc_err': tc_err_esperado,
                'origem': origem
            })
    return eventos

# ============================================================
# 3. ANÁLISE COMPARATIVA DE EFEMÉRIDES
# ============================================================
def analisar_efemerides(ephemerides_list, t, residual_manchas, transitos_conhecidos_b, 
                        RP_b, A_b, INC_b, janela_dias=0.25):
    """
    Compara múltiplas efemérides nos mesmos trânsitos
    """
    
    # Mapear eventos para a primeira efeméride como referência
    eventos_ref = mapear_transitos_hibrido(ephemerides_list[0], t, janela_dias, transitos_conhecidos_b)
    
    resultados = {eph.name: [] for eph in ephemerides_list}
    
    for ev in eventos_ref:
        epoch = ev['epoch']
        
        # Janela de dados
        janela = (t >= ev['tc_esperado'] - janela_dias) & (t <= ev['tc_esperado'] + janela_dias)
        t_f, fluxo_f = t[janela], residual_manchas[janela]
        
        if len(t_f) < 5:
            continue
        
        # Para cada efeméride, calcular o centro esperado e fazer fit
        for eph in ephemerides_list:
            tc_esperado_eph = eph.T0 + epoch * eph.P
            
            def wrapper_ajuste(t_j, t0_fit):
                return modelo_ajuste_t0(t_j, t0_fit, eph.P, RP_b, A_b, INC_b)
            
            try:
                popt, pcov = curve_fit(wrapper_ajuste, t_f, fluxo_f, 
                                      p0=[tc_esperado_eph], 
                                      bounds=(tc_esperado_eph-0.08, tc_esperado_eph+0.08),
                                      maxfev=5000)
                tc_medido, tc_err_medido = popt[0], np.sqrt(pcov[0,0])
            except:
                tc_medido, tc_err_medido = tc_esperado_eph, eph.T0_err
            
            # O-C: Diferença entre medido e teórico puro
            tc_teorico = eph.T0 + epoch * eph.P
            o_c_segundos = (tc_medido - tc_teorico) * 24 * 3600
            
            resultados[eph.name].append({
                'epoch': epoch,
                'tc_esperado': tc_esperado_eph,
                'tc_medido': tc_medido,
                'tc_err': tc_err_medido,
                'o_c_s': o_c_segundos,
                'fluxo': fluxo_f,
                'tempo': t_f
            })
    
    return eventos_ref, resultados

# ============================================================
# 4. GERAÇÃO DE DADOS (simulado para exemplo)
# ============================================================
# NOTA: Você precisa substituir isso pelos seus dados reais de t e residual_manchas
# Para exemplo, criando dados sintéticos:

print("\n[NOTA] Usando dados simulados. Substitua 't' e 'residual_manchas' pelos seus dados reais.\n")

# Gerar dados sintéticos de exemplo
np.random.seed(42)
eventos_exemplo = [0, 2, 84, 85, 86]
t_lista = []
residual_lista = []

for ep in eventos_exemplo:
    tc_base = eph_old.T0 + ep * eph_old.P
    t_evento = np.linspace(tc_base - 0.25, tc_base + 0.25, 300)
    
    # Modelo de trânsito
    flux_modelo = modelo_ajuste_t0(t_evento, tc_base, eph_old.P, RP_b, A_b, INC_b)
    # Adicionar ruído
    flux_noisy = flux_modelo + np.random.normal(0, 0.002, len(t_evento))
    
    t_lista.extend(t_evento)
    residual_lista.extend(flux_noisy)

t = np.array(t_lista)
residual_manchas = np.array(residual_lista)

# ============================================================
# 5. ANÁLISE COMPARATIVA
# ============================================================
eventos_ref, resultados = analisar_efemerides(
    [eph_old, eph_new], t, residual_manchas, 
    transitos_conhecidos_b, RP_b, A_b, INC_b
)

# ============================================================
# 6. PLOTAGEM: Gráficos Individuais com Comparação
# ============================================================
%matplotlib qt

n_plots = len(eventos_ref)
if n_plots > 0:
    fig, axes = plt.subplots(2, n_plots, figsize=(4 * n_plots, 8), sharey='row')
    if n_plots == 1: 
        axes = axes.reshape(2, 1)
    
    for i, ev in enumerate(eventos_ref):
        epoch = ev['epoch']
        
        # --- Linha 1: Comparação Visual dos Ajustes ---
        ax = axes[0, i]
        
        for j, eph in enumerate([eph_old, eph_new]):
            res = resultados[eph.name][i]
            
            # Plotar dados
            ax.plot(res['tempo'], res['fluxo'], 'k.', ms=2, alpha=0.3)
            
            # Plotar modelo com T_c medido
            modelo_fit = modelo_ajuste_t0(res['tempo'], res['tc_medido'], 
                                         eph.P, RP_b, A_b, INC_b)
            cor = 'crimson' if eph.name == 'OLD' else 'dodgerblue'
            estilo = '--' if eph.name == 'NEW' else '-'
            ax.plot(res['tempo'], modelo_fit, color=cor, linestyle=estilo, 
                   lw=2, label=f"{eph.name}")
        
        ax.set_title(f"Época {epoch}", fontsize=11, fontweight='bold')
        ax.grid(alpha=0.2)
        ax.legend(loc='lower center', fontsize=9)
        if i == 0:
            ax.set_ylabel('Fluxo Residual')
        
        # --- Linha 2: O-C Comparison ---
        ax = axes[1, i]
        
        o_c_old = resultados['OLD'][i]['o_c_s']
        o_c_new = resultados['NEW'][i]['o_c_s']
        
        x_pos = [0, 1]
        o_c_values = [o_c_old, o_c_new]
        cores = ['crimson', 'dodgerblue']
        
        bars = ax.bar(x_pos, o_c_values, color=cores, alpha=0.7, width=0.5)
        
        # Adicionar valores nas barras
        for bar, val in zip(bars, o_c_values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.1f}s',
                   ha='center', va='bottom' if height > 0 else 'top', fontsize=9)
        
        ax.axhline(0, color='k', linestyle='-', linewidth=0.8, alpha=0.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(['OLD', 'NEW'])
        ax.set_ylabel('O-C [segundos]')
        ax.grid(alpha=0.2, axis='y')
    
    plt.suptitle('AU Mic b - Comparação Efemérides OLD vs NEW', fontweight='bold', fontsize=13, y=1.00)
    plt.tight_layout()
    plt.show()

# ============================================================
# 7. GRÁFICO EMPILHADO (Stacked Transits)
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for j, eph in enumerate([eph_old, eph_new]):
    ax = ax1 if j == 0 else ax2
    
    # Normalizar todos os trânsitos à fase 0
    for i, ev in enumerate(eventos_ref):
        res = resultados[eph.name][i]
        
        # Fase folded: tempo relativo ao centro do trânsito
        fase = (res['tempo'] - res['tc_medido']) * 24  # em horas
        
        # Normalizar fluxo: subtrair 1 para deixar residual claro
        fluxo_norm = res['fluxo'] - 1.0
        
        # Plot com transparência
        ax.plot(fase, fluxo_norm, 'o-', markersize=2, linewidth=1, 
               alpha=0.4, label=f"Época {res['epoch']}")
    
    ax.axvline(0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Centro (T_c)')
    ax.set_xlabel('Fase [horas]')
    ax.set_ylabel('Fluxo Residual (norm)')
    ax.set_title(f"Trânsitos Empilhados - Efeméride {eph.name}\n({eph})", fontsize=11)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8, loc='best')

plt.tight_layout()
plt.show()

# ============================================================
# 8. TABELA RESUMO
# ============================================================
print("\n" + "="*90)
print("RESUMO DE RESULTADOS: O-C em Segundos para Cada Época")
print("="*90)
print(f"{'Época':^8} | {'OLD O-C (s)':^15} | {'NEW O-C (s)':^15} | {'Δ O-C (s)':^15}")
print("-"*90)

for i, ev in enumerate(eventos_ref):
    if i < len(resultados['OLD']):
        o_c_old = resultados['OLD'][i]['o_c_s']
        o_c_new = resultados['NEW'][i]['o_c_s']
        delta_oc = o_c_old - o_c_new
        
        print(f"{ev['epoch']:^8} | {o_c_old:^15.2f} | {o_c_new:^15.2f} | {delta_oc:^15.2f}")

print("="*90)


PASSO 3: AJUSTE FOTOMÉTRICO COM DADOS DE LITERATURA (AU Mic b)
COMPARAÇÃO EFEMÉRIDES OLD vs NEW

OLD: T0=1330.390510±0.000160, P=8.4631516
NEW: T0=1330.380549±0.000431, P=8.4631516

Diferença ΔT0 (OLD - NEW): 860.630 segundos

[NOTA] Usando dados simulados. Substitua 't' e 'residual_manchas' pelos seus dados reais.


RESUMO DE RESULTADOS: O-C em Segundos para Cada Época
 Época   |   OLD O-C (s)   |   NEW O-C (s)   |    Δ O-C (s)   
------------------------------------------------------------------------------------------
   0     |      23.33      |     883.32      |     -859.99    
   2     |     -202.62     |     658.02      |     -860.64    
   84    |      75.62      |     936.24      |     -860.61    
   85    |     129.27      |     989.52      |     -860.25    
   86    |      -2.50      |     859.29      |     -861.78    


In [13]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, minimize
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks

# =====================================================================
# 1. ENTRADA DE DADOS (Completa com todos os trânsitos disponíveis)
# =====================================================================
print("\n" + "="*70)
print("ANÁLISE DE MÁXIMA PRECISÃO: EFEMÉRIDE AU MIC B")
print("="*70)

# Épocas e medições documentadas de AU Mic b
epocas = np.array([
    0,      # Setor 1 (Lit.)
    2,      # Setor 1 (Lit.)
    84,     # Setor 27 (Lit.)
    85,     # Setor 27 (Lit.)
    86,     # Setor 27 (Lit.)
    302,    # Setor 95 (Novo)
    304     # Setor 95 (Novo)
], dtype=float)

# Tc medidos (BTJD) - seus valores ajustados
tc_medidos = np.array([
    1330.38596,
    1347.31851,
    2041.28174,
    2049.74505,
    2058.17115,
    3886.26528,
    3903.18781
])

# Incertezas (1-sigma)
tc_erros = np.array([
    0.00063,
    0.00095,
    0.00059,
    0.00084,
    0.00136,
    0.00137,
    0.00164
])

# =====================================================================
# 2. MODELOS DE EFEMÉRIDE
# =====================================================================

def modelo_linear(E, T0, P):
    """Modelo linear simples: T0 + E*P"""
    return T0 + E * P

def modelo_quadratico(E, T0, P, P_dot):
    """Modelo quadrático com aceleração: T0 + E*P + 0.5*P_dot*E²"""
    return T0 + E * P + 0.5 * P_dot * E**2

def modelo_linear_ttv(E, T0, P, A_ttv, f_ttv):
    """Modelo linear + TTV sinusoidal: para detectar planeta c"""
    return T0 + E * P + A_ttv * np.sin(2 * np.pi * f_ttv * E)

# =====================================================================
# 3. AJUSTE LINEAR PONDERADO (WLS - Weighted Least Squares)
# =====================================================================
print("\n[1] AJUSTE LINEAR PONDERADO (WLS)")
print("-" * 70)

popt_linear, pcov_linear = curve_fit(
    modelo_linear, 
    epocas, 
    tc_medidos, 
    p0=[1330.39051, 8.463000],
    sigma=tc_erros, 
    absolute_sigma=True
)

T0_linear, P_linear = popt_linear
T0_linear_err, P_linear_err = np.sqrt(np.diag(pcov_linear))

tc_calc_linear = modelo_linear(epocas, T0_linear, P_linear)
oc_linear = tc_medidos - tc_calc_linear
oc_linear_min = oc_linear * 24.0 * 60.0
chi2_linear = np.sum((oc_linear / tc_erros)**2)
chi2_red_linear = chi2_linear / (len(epocas) - 2)

print(f"T0 (Época Ref.)  : {T0_linear:.6f} ± {T0_linear_err:.6f} BTJD")
print(f"P (Período)      : {P_linear:.7f} ± {P_linear_err:.7f} dias")
print(f"χ²_red           : {chi2_red_linear:.3f}")
print(f"Resíduos RMS     : {np.std(oc_linear_min):.3f} minutos")

# =====================================================================
# 4. AJUSTE QUADRÁTICO (Detecta aceleração/desaceleração orbital)
# =====================================================================
print("\n[2] AJUSTE QUADRÁTICO (Detecta P-dot)")
print("-" * 70)

popt_quad, pcov_quad = curve_fit(
    modelo_quadratico,
    epocas,
    tc_medidos,
    p0=[T0_linear, P_linear, 0],  # P_dot ≈ 0 inicialmente
    sigma=tc_erros,
    absolute_sigma=True,
    maxfev=5000
)

T0_quad, P_quad, P_dot = popt_quad
errs_quad = np.sqrt(np.diag(pcov_quad))
T0_quad_err, P_quad_err, P_dot_err = errs_quad

tc_calc_quad = modelo_quadratico(epocas, T0_quad, P_quad, P_dot)
oc_quad = tc_medidos - tc_calc_quad
oc_quad_min = oc_quad * 24.0 * 60.0
chi2_quad = np.sum((oc_quad / tc_erros)**2)
chi2_red_quad = chi2_quad / (len(epocas) - 3)

print(f"T0 (Época Ref.)  : {T0_quad:.6f} ± {T0_quad_err:.6f} BTJD")
print(f"P (Período)      : {P_quad:.7f} ± {P_quad_err:.7f} dias")
print(f"P_dot (Aceler.)  : {P_dot:.2e} ± {P_dot_err:.2e} dias/ciclo")
if P_dot != 0:
    print(f"  → Significância : {abs(P_dot)/P_dot_err:.2f}σ")
print(f"χ²_red           : {chi2_red_quad:.3f}")
print(f"Resíduos RMS     : {np.std(oc_quad_min):.3f} minutos")
print(f"ΔX²_red (vs Lin.): {chi2_red_quad - chi2_red_linear:.3f}")

# =====================================================================
# 5. ANÁLISE FOURIER DOS RESÍDUOS (Detecta TTV periódicas)
# =====================================================================
print("\n[3] ANÁLISE FOURIER DO RESÍDUO LINEAR (Detecta TTV)")
print("-" * 70)

# Usar resíduos do ajuste linear para buscar periodicidades
fft_result = np.abs(fft(oc_linear))
freqs = fftfreq(len(epocas), epocas[1] - epocas[0] if len(epocas) > 1 else 1.0)

# Pegar apenas frequências positivas
pos_freqs = freqs[:len(freqs)//2]
pos_fft = fft_result[:len(freqs)//2]

# Encontrar picos (potenciais periodicidades)
peaks, props = find_peaks(pos_fft[1:], height=np.max(pos_fft)/3)  # 1/3 da altura máxima
peaks = peaks + 1  # Ajustar índice

if len(peaks) > 0:
    top_peak_idx = peaks[np.argmax(props['peak_heights'])]
    freq_ttv = pos_freqs[top_peak_idx]
    periodo_ttv = 1.0 / freq_ttv if freq_ttv != 0 else np.inf
    amplitude_fft = pos_fft[top_peak_idx]
    
    print(f"Frequência detectada  : {freq_ttv:.6f} ciclos⁻¹")
    print(f"Período TTV           : {periodo_ttv:.1f} épocas")
    print(f"Amplitude (FFT)       : {amplitude_fft:.6f}")
else:
    print("Nenhuma periodicidade detectada (ruído aleatório dominante)")
    freq_ttv = 0
    periodo_ttv = np.inf

# =====================================================================
# 6. AJUSTE COM TTV SINUSOIDAL (Detecta Planeta c)
# =====================================================================
if freq_ttv != 0 and periodo_ttv < 200:
    print("\n[4] AJUSTE COM TTV SINUSOIDAL (Possível Planeta c)")
    print("-" * 70)
    
    try:
        popt_ttv, pcov_ttv = curve_fit(
            modelo_linear_ttv,
            epocas,
            tc_medidos,
            p0=[T0_linear, P_linear, 0.001, freq_ttv],
            sigma=tc_erros,
            absolute_sigma=True,
            maxfev=5000
        )
        
        T0_ttv, P_ttv, A_ttv, f_ttv = popt_ttv
        errs_ttv = np.sqrt(np.diag(pcov_ttv))
        
        tc_calc_ttv = modelo_linear_ttv(epocas, T0_ttv, P_ttv, A_ttv, f_ttv)
        oc_ttv = tc_medidos - tc_calc_ttv
        oc_ttv_min = oc_ttv * 24.0 * 60.0
        chi2_ttv = np.sum((oc_ttv / tc_erros)**2)
        chi2_red_ttv = chi2_ttv / (len(epocas) - 4)
        
        print(f"T0               : {T0_ttv:.6f} ± {errs_ttv[0]:.6f} BTJD")
        print(f"P                : {P_ttv:.7f} ± {errs_ttv[1]:.7f} dias")
        print(f"A_TTV            : {A_ttv:.6f} ± {errs_ttv[2]:.6f} dias ({A_ttv*24*60:.2f} min)")
        print(f"f_TTV            : {f_ttv:.6f} ± {errs_ttv[3]:.6f} ciclos⁻¹")
        print(f"  → Período      : {1.0/f_ttv:.1f} épocas")
        print(f"χ²_red           : {chi2_red_ttv:.3f}")
        print(f"Resíduos RMS     : {np.std(oc_ttv_min):.3f} minutos")
        print(f"ΔX²_red (vs Lin.): {chi2_red_ttv - chi2_red_linear:.3f}")
        
        modelo_melhor = "TTV"
        popt_melhor = popt_ttv
        oc_melhor = oc_ttv_min
        chi2_red_melhor = chi2_red_ttv
        
    except Exception as e:
        print(f"Erro no ajuste TTV: {e}")
        modelo_melhor = "Quadrático" if abs(P_dot) > 3*P_dot_err else "Linear"
        popt_melhor = popt_quad if modelo_melhor == "Quadrático" else popt_linear
        oc_melhor = oc_quad_min if modelo_melhor == "Quadrático" else oc_linear_min
        chi2_red_melhor = chi2_red_quad if modelo_melhor == "Quadrático" else chi2_red_linear
else:
    modelo_melhor = "Quadrático" if abs(P_dot) > 3*P_dot_err else "Linear"
    popt_melhor = popt_quad if modelo_melhor == "Quadrático" else popt_linear
    oc_melhor = oc_quad_min if modelo_melhor == "Quadrático" else oc_linear_min
    chi2_red_melhor = chi2_red_quad if modelo_melhor == "Quadrático" else chi2_red_linear

# =====================================================================
# 7. ESTIMATIVA DO MELHOR MODELO
# =====================================================================
print("\n" + "="*70)
print(f"MODELO RECOMENDADO: {modelo_melhor} (χ²_red = {chi2_red_melhor:.3f})")
print("="*70)

if modelo_melhor == "Linear":
    print(f"\nEFEMÉRIDE FINAL (LINEAR):")
    print(f"  T0 = {T0_linear:.6f} ± {T0_linear_err:.6f} BTJD")
    print(f"  P  = {P_linear:.7f} ± {P_linear_err:.7f} dias")
elif modelo_melhor == "Quadrático":
    print(f"\nEFEMÉRIDE FINAL (QUADRÁTICA):")
    print(f"  T0    = {T0_quad:.6f} ± {T0_quad_err:.6f} BTJD")
    print(f"  P     = {P_quad:.7f} ± {P_quad_err:.7f} dias")
    print(f"  P_dot = {P_dot:.2e} ± {P_dot_err:.2e} dias/ciclo")
else:
    print(f"\nEFEMÉRIDE FINAL (COM TTV):")
    print(f"  T0    = {T0_ttv:.6f} BTJD")
    print(f"  P     = {P_ttv:.7f} dias")
    print(f"  A_TTV = {A_ttv*24*60:.3f} minutos")
    print(f"  f_TTV = {1.0/f_ttv:.1f} épocas (período)")

# =====================================================================
# 8. TABELA COMPLETA DE RESÍDUOS
# =====================================================================
print("\n" + "="*70)
print("TABELA DE RESÍDUOS O-C (Usando Melhor Modelo)")
print("="*70)
print(f"{'Época':<8}{'T_c Med.(BTJD)':<18}{'O-C (min)':<12}{'Erro (min)':<12}{'σ (desvios)':<12}")
print("-"*70)

for i, (ep, tc_med, err) in enumerate(zip(epocas, tc_medidos, tc_erros)):
    o_c_m = oc_melhor[i]
    sigma_desvios = o_c_m / (err * 24 * 60)
    print(f"{int(ep):<8}{tc_med:<18.5f}{o_c_m:<+12.3f}{err*24*60:<12.3f}{sigma_desvios:<+12.2f}")

print(f"\nResíduo RMS Total: {np.sqrt(np.sum((oc_melhor / (tc_erros * 24 * 60))**2) / len(epocas)):.3f} σ")

# =====================================================================
# 9. PREVISÃO PARA PRÓXIMOS TRÂNSITOS
# =====================================================================
print("\n" + "="*70)
print("PREVISÕES PARA PRÓXIMOS TRÂNSITOS")
print("="*70)

epocas_futuro = np.array([305, 306, 307, 400, 500])

print(f"\nUsando efeméride: {modelo_melhor}\n")
print(f"{'Época':<8}{'Tempo Previsto (BTJD)':<28}{'Data aprox. (2020 + dias)':<20}")
print("-"*70)

for ep_fut in epocas_futuro:
    if modelo_melhor == "Linear":
        tc_pred = T0_linear + ep_fut * P_linear
        err_pred = np.sqrt(T0_linear_err**2 + (ep_fut * P_linear_err)**2)
    elif modelo_melhor == "Quadrático":
        tc_pred = T0_quad + ep_fut * P_quad + 0.5 * P_dot * ep_fut**2
        err_pred = np.sqrt(T0_quad_err**2 + (ep_fut * P_quad_err)**2)
    else:
        tc_pred = T0_ttv + ep_fut * P_ttv + A_ttv * np.sin(2*np.pi*f_ttv*ep_fut)
        err_pred = np.sqrt(errs_ttv[0]**2 + (ep_fut * errs_ttv[1])**2)
    
    # Converter BTJD para data aproximada (BTJD = BJD - 2457000)
    bjd = tc_pred + 2457000
    dias_2020 = bjd - 2458849  # 2020-01-01 ≈ BJD 2458849
    
    tc_str = f"{tc_pred:.5f} ± {err_pred:.5f}"
    print(f"{int(ep_fut):<8}{tc_str:<28}{dias_2020:<20.1f} dias")

# =====================================================================
# 10. PLOTAGENS
# =====================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Gráfico 1: Modelos Comparados ---
ax = axes[0, 0]
tc_plot = np.linspace(epocas.min() - 10, epocas.max() + 10, 500)
ax.plot(tc_plot, modelo_linear(tc_plot, T0_linear, P_linear), 'b-', lw=2, label='Linear')
ax.plot(tc_plot, modelo_quadratico(tc_plot, T0_quad, P_quad, P_dot), 'g--', lw=2, label='Quadrático')
if modelo_melhor == "TTV":
    ax.plot(tc_plot, modelo_linear_ttv(tc_plot, T0_ttv, P_ttv, A_ttv, f_ttv), 'r-.', lw=2, label='Com TTV')

ax.plot(epocas, tc_medidos, 'ko', markersize=8, label='Observado')
ax.set_xlabel('Época')
ax.set_ylabel('T_c (BTJD)')
ax.set_title('Modelos de Efeméride Comparados')
ax.legend()
ax.grid(alpha=0.3)

# --- Gráfico 2: Resíduos O-C (Linear) ---
ax = axes[0, 1]
mask_old = epocas < 100
ax.errorbar(epocas[mask_old], oc_linear_min[mask_old], yerr=tc_erros[mask_old]*24*60,
            fmt='o', color='black', label='Setores 1 & 27', capsize=3, ms=6)
ax.errorbar(epocas[~mask_old], oc_linear_min[~mask_old], yerr=tc_erros[~mask_old]*24*60,
            fmt='s', color='crimson', label='Setor 95', capsize=3, ms=7)
ax.axhline(0, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Época')
ax.set_ylabel('O-C (minutos)')
ax.set_title(f'Resíduos Modelo Linear (χ²_red = {chi2_red_linear:.3f})')
ax.legend()
ax.grid(alpha=0.3)

# --- Gráfico 3: Resíduos O-C (Melhor Modelo) ---
ax = axes[1, 0]
mask_old = epocas < 100
ax.errorbar(epocas[mask_old], oc_melhor[mask_old], yerr=tc_erros[mask_old]*24*60,
            fmt='o', color='darkblue', label='Setores 1 & 27', capsize=3, ms=6)
ax.errorbar(epocas[~mask_old], oc_melhor[~mask_old], yerr=tc_erros[~mask_old]*24*60,
            fmt='s', color='darkred', label='Setor 95', capsize=3, ms=7)
ax.axhline(0, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Época')
ax.set_ylabel('O-C (minutos)')
ax.set_title(f'Resíduos Modelo {modelo_melhor} (χ²_red = {chi2_red_melhor:.3f})')
ax.legend()
ax.grid(alpha=0.3)

# --- Gráfico 4: Transformada de Fourier ---
ax = axes[1, 1]
ax.semilogy(pos_freqs[1:], pos_fft[1:], 'b-', lw=1.5)
if len(peaks) > 0:
    ax.semilogy(pos_freqs[top_peak_idx], pos_fft[top_peak_idx], 'ro', markersize=10, 
               label=f'Período = {periodo_ttv:.1f} épocas')
    ax.legend()
ax.set_xlabel('Frequência (ciclos⁻¹)')
ax.set_ylabel('Amplitude FFT')
ax.set_title('Transformada de Fourier dos Resíduos (Busca de TTV)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("Análise concluída com sucesso!")
print("="*70 + "\n")



ANÁLISE DE MÁXIMA PRECISÃO: EFEMÉRIDE AU MIC B

[1] AJUSTE LINEAR PONDERADO (WLS)
----------------------------------------------------------------------
T0 (Época Ref.)  : 1330.380549 ± 0.000431 BTJD
P (Período)      : 8.4631516 ± 0.0000039 dias
χ²_red           : 256.264
Resíduos RMS     : 24.741 minutos

[2] AJUSTE QUADRÁTICO (Detecta P-dot)
----------------------------------------------------------------------
T0 (Época Ref.)  : 1330.388043 ± 0.000530 BTJD
P (Período)      : 8.4628985 ± 0.0000111 dias
P_dot (Aceler.)  : 1.76e-06 ± 7.21e-08 dias/ciclo
  → Significância : 24.34σ
χ²_red           : 172.123
Resíduos RMS     : 17.764 minutos
ΔX²_red (vs Lin.): -84.141

[3] ANÁLISE FOURIER DO RESÍDUO LINEAR (Detecta TTV)
----------------------------------------------------------------------
Nenhuma periodicidade detectada (ruído aleatório dominante)

MODELO RECOMENDADO: Quadrático (χ²_red = 172.123)

EFEMÉRIDE FINAL (QUADRÁTICA):
  T0    = 1330.388043 ± 0.000530 BTJD
  P     = 8.4628985 

In [64]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# =====================================================================
# 1. FUNÇÃO MATEMÁTICA DO TRÂNSITO
# =====================================================================
def modelo_transito(t, tc, profundidade, largura, linha_base):
    """
    Aproxima o trânsito planetário como uma Gaussiana invertida.
    Ótimo para encontrar o centro (Tc) exato em curvas de luz com ruído.
    """
    return linha_base - profundidade * np.exp(-0.5 * ((t - tc) / largura)**2)

# =====================================================================
# 2. SIMULAÇÃO DE DADOS BRUTOS (Substitua pelos seus dados do TESS)
# =====================================================================
# Aqui simulamos uma janela de tempo ao redor do trânsito esperado
tempo_estimado = 1330.38500 # Estimativa visual ou de catálogo
tempo_bruto = np.linspace(tempo_estimado - 0.2, tempo_estimado + 0.2, 300)

# Simulamos o fluxo com um trânsito ocorrendo em 1330.38596
fluxo_verdadeiro = modelo_transito(tempo_bruto, tc=1330.38596, profundidade=0.01, largura=0.03, linha_base=1.0)

# Adicionamos ruído gaussiano típico de telescópios espaciais
np.random.seed(42)
ruido = np.random.normal(0, 0.0015, len(tempo_bruto))
fluxo_observado = fluxo_verdadeiro + ruido
erro_fluxo = np.full_like(fluxo_observado, 0.0015)

# =====================================================================
# 3. AJUSTE DO MODELO AOS DADOS BRUTOS
# =====================================================================
# Chutes iniciais (p0) para ajudar o algoritmo a convergir
chute_tc = tempo_estimado
chute_profundidade = 1.0 - np.min(fluxo_observado)
chute_largura = 0.03
chute_base = 1.0
chutes_iniciais = [chute_tc, chute_profundidade, chute_largura, chute_base]

# Limites para evitar que o ajuste fuja do controle (bounds)
# (tc_min, tc_max), (prof_min, prof_max), (larg_min, larg_max), (base_min, base_max)
limites_inf = [tempo_estimado - 0.1, 0.0, 0.001, 0.9]
limites_sup = [tempo_estimado + 0.1, 0.1, 0.100, 1.1]

# Ajuste da curva (Curve Fit)
popt, pcov = curve_fit(
    modelo_transito, 
    tempo_bruto, 
    fluxo_observado, 
    p0=chutes_iniciais,
    sigma=erro_fluxo,          # Pondera o ajuste pelo erro instrumental
    absolute_sigma=True,
    bounds=(limites_inf, limites_sup)
)

# Extraindo os resultados
tc_medido, profundidade_medida, largura_medida, base_medida = popt
erros = np.sqrt(np.diag(pcov)) # Incerteza (1-sigma)
erro_tc = erros[0]

# =====================================================================
# 4. RESULTADOS E PREPARAÇÃO PARA O CÓDIGO O-C
# =====================================================================
print("\n" + "="*50)
print("RESULTADO DA MEDIÇÃO DO TRÂNSITO INDIVIDUAL")
print("="*50)
print(f"Tc Calculado     : {tc_medido:.5f} BTJD")
print(f"Incerteza do Tc  : ± {erro_tc:.5f} BTJD")
print(f"Profundidade     : {profundidade_medida*100:.2f}%")
print("="*50)

print("\n** Copie estes valores para o seu script de Efeméride (O-C): **")
print(f"tc_medidos.append({tc_medido:.5f})")
print(f"tc_erros.append({erro_tc:.5f})")

# =====================================================================
# 5. VISUALIZAÇÃO DO AJUSTE
# =====================================================================
plt.figure(figsize=(10, 6))

# Dados brutos
plt.errorbar(tempo_bruto, fluxo_observado, yerr=erro_fluxo, fmt='o', color='gray', 
             alpha=0.5, markersize=4, label='Dados Brutos (Fluxo)')

# Modelo ajustado
tempo_fino = np.linspace(min(tempo_bruto), max(tempo_bruto), 1000)
fluxo_ajustado = modelo_transito(tempo_fino, tc_medido, profundidade_medida, largura_medida, base_medida)
plt.plot(tempo_fino, fluxo_ajustado, 'r-', linewidth=2, label=f'Modelo Ajustado')

# Linha marcando o Tc
plt.axvline(tc_medido, color='blue', linestyle='--', label=f'Centro (Tc): {tc_medido:.5f}')

plt.xlabel('Tempo (BTJD)', fontsize=12)
plt.ylabel('Fluxo Relativo', fontsize=12)
plt.title('Ajuste de Trânsito Individual para Extração do Tempo Central', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


RESULTADO DA MEDIÇÃO DO TRÂNSITO INDIVIDUAL
Tc Calculado     : 1330.38503 BTJD
Incerteza do Tc  : ± 0.00100 BTJD
Profundidade     : 0.99%

** Copie estes valores para o seu script de Efeméride (O-C): **
tc_medidos.append(1330.38503)
tc_erros.append(0.00100)


In [62]:
# ============================================================
# COMPARAÇÃO EFEMÉRIDE OLD VS NOVA - AU Mic b
# COM ERROS DE T0
# ============================================================

%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt


print("\n" + "="*70)
print("COMPARAÇÃO DAS EFEMÉRIDES AU Mic b")
print("="*70)


# ============================================================
# EFEMÉRIDE OLD
# ============================================================

T0_old = 1330.39051
erro_T0_old = 0.00016

# colocar aqui o período antigo quando disponível
P_old = 8.4631516


# ============================================================
# EFEMÉRIDE NOVA
# ============================================================

T0_new = 1330.390466
erro_T0_new = 0.000146

P_new = 8.4631516


print("\nEFEMÉRIDE OLD")
print("-"*40)
print(f"T0 = {T0_old:.6f} ± {erro_T0_old:.6f} BTJD")
print(f"P  = {P_old:.7f} dias")
print(f"Erro T0 = {erro_T0_old*24*60:.2f} minutos")


print("\nEFEMÉRIDE NOVA")
print("-"*40)
print(f"T0 = {T0_new:.6f} ± {erro_T0_new:.6f} BTJD")
print(f"P  = {P_new:.7f} dias")
print(f"Erro T0 = {erro_T0_new*24*60:.2f} minutos")


# diferença entre efemérides

delta = (T0_old - T0_new)*24*60

print("\nDiferença entre T0:")
print(f"{delta:.2f} minutos")


# ============================================================
# INTERVALO DOS DADOS
# ============================================================

tempo_min = np.min([np.min(t) for t in t_todos])
tempo_max = np.max([np.max(t) for t in t_todos])


# ============================================================
# CALCULA TRÂNSITOS OLD
# ============================================================

nmin_old = int(np.floor((tempo_min-T0_old)/P_old))
nmax_old = int(np.ceil((tempo_max-T0_old)/P_old))

ep_old = np.arange(
    nmin_old,
    nmax_old+1
)

trans_old = T0_old + ep_old*P_old



# ============================================================
# CALCULA TRÂNSITOS NOVA
# ============================================================

nmin_new = int(np.floor((tempo_min-T0_new)/P_new))
nmax_new = int(np.ceil((tempo_max-T0_new)/P_new))

ep_new = np.arange(
    nmin_new,
    nmax_new+1
)

trans_new = T0_new + ep_new*P_new



# ============================================================
# MOSTRA LISTA DOS TRÂNSITOS
# ============================================================

print("\n")
print("="*70)
print("TRÂNSITOS PREVISTOS")
print("="*70)


print("\nOLD")
for e,t in zip(ep_old,trans_old):
    print(f"Época {e:4d}  {t:.6f} BTJD")


print("\nNOVA")
for e,t in zip(ep_new,trans_new):
    print(f"Época {e:4d}  {t:.6f} BTJD")



# ============================================================
# NOMES DOS SETORES
# ============================================================

setor_nomes = [
    "Setor 1 (2018)",
    "Setor 27 (2020)",
    "Setor 95 (2025)"
]



# ============================================================
# FIGURA
# ============================================================

fig, axes = plt.subplots(
    3,
    1,
    figsize=(14,10)
)



for i,(t_sec,f_sec) in enumerate(zip(t_todos,f_todos)):

    ax = axes[i]


    # dados

    ax.plot(
        t_sec,
        f_sec,
        'k.',
        markersize=2,
        alpha=0.5,
        label="Dados TESS"
    )



    # ========================================================
    # OLD
    # ========================================================

    first_old=True

    for e,tr in zip(ep_old,trans_old):

        if t_sec[0] <= tr <= t_sec[-1]:


            ax.axvline(
                tr,
                color='blue',
                linestyle='--',
                linewidth=1.8,
                label="Efeméride OLD" if first_old else None
            )


            ax.axvspan(
                tr-erro_T0_old,
                tr+erro_T0_old,
                color='blue',
                alpha=0.15,
                label="Erro OLD (1σ)" if first_old else None
            )


            ax.text(
                tr,
                np.nanmax(f_sec),
                f"E{e}",
                color="blue",
                rotation=90,
                fontsize=8
            )


            first_old=False



    # ========================================================
    # NOVA
    # ========================================================

    first_new=True

    for e,tr in zip(ep_new,trans_new):

        if t_sec[0] <= tr <= t_sec[-1]:


            ax.axvline(
                tr,
                color='red',
                linestyle='-',
                linewidth=1.8,
                label="Efeméride NOVA" if first_new else None
            )


            ax.axvspan(
                tr-erro_T0_new,
                tr+erro_T0_new,
                color='red',
                alpha=0.15,
                label="Erro NOVA (1σ)" if first_new else None
            )


            ax.text(
                tr,
                np.nanmin(f_sec),
                f"E{e}",
                color="red",
                rotation=90,
                fontsize=8
            )


            first_new=False



    # ========================================================
    # FORMATAÇÃO
    # ========================================================

    ax.set_title(
        setor_nomes[i],
        fontsize=13,
        fontweight='bold'
    )


    ax.set_ylabel(
        "Fluxo normalizado"
    )


    ax.grid(
        alpha=0.3,
        linestyle=":"
    )


    ax.legend(
        fontsize=9,
        loc="best"
    )



axes[-1].set_xlabel(
    "Tempo [BTJD]",
    fontsize=12
)



plt.suptitle(
    "AU Mic b - Comparação Efeméride OLD vs NOVA\n"
    "Azul = OLD | Vermelho = NOVA",
    fontsize=15,
    fontweight="bold"
)


plt.tight_layout()

plt.show()



print("\n✓ Comparação finalizada!")


COMPARAÇÃO DAS EFEMÉRIDES AU Mic b

EFEMÉRIDE OLD
----------------------------------------
T0 = 1330.390510 ± 0.000160 BTJD
P  = 8.4631516 dias
Erro T0 = 0.23 minutos

EFEMÉRIDE NOVA
----------------------------------------
T0 = 1330.390466 ± 0.000146 BTJD
P  = 8.4631516 dias
Erro T0 = 0.21 minutos

Diferença entre T0:
0.06 minutos


TRÂNSITOS PREVISTOS

OLD
Época   -1  1321.927358 BTJD
Época    0  1330.390510 BTJD
Época    1  1338.853662 BTJD
Época    2  1347.316813 BTJD
Época    3  1355.779965 BTJD
Época    4  1364.243116 BTJD
Época    5  1372.706268 BTJD
Época    6  1381.169420 BTJD
Época    7  1389.632571 BTJD
Época    8  1398.095723 BTJD
Época    9  1406.558874 BTJD
Época   10  1415.022026 BTJD
Época   11  1423.485178 BTJD
Época   12  1431.948329 BTJD
Época   13  1440.411481 BTJD
Época   14  1448.874632 BTJD
Época   15  1457.337784 BTJD
Época   16  1465.800936 BTJD
Época   17  1474.264087 BTJD
Época   18  1482.727239 BTJD
Época   19  1491.190390 BTJD
Época   20  1499.653542 BTJD


In [60]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import batman
import emcee
from multiprocessing import Pool
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*60)
print("PASSO 3: AJUSTE FOTOMÉTRICO COM MCMC (AU Mic b)")
print("="*60)

# ============================================================
# 1. DICIONÁRIO DE TRÂNSITOS CONHECIDOS (Setor 1 e 27)
# ============================================================
transitos_conhecidos_b = {
    0:  {'tc': 1330.39046, 'err': 0.00016},
    2:  {'tc': 1347.31646, 'err': 0.00016},
    84: {'tc': 2041.28238, 'err': 0.00026},
    85: {'tc': 2049.74538, 'err': 0.00026},
    86: {'tc': 2058.20838, 'err': 0.00026}
}

# Constantes do Planeta b
T0_b, T0_b_err = 1330.39051, 0.00015
P_b, P_b_err   = 8.463000, 0.000002
RP_b, A_b, INC_b = 0.0526, 19.1, 89.5
LDC = [0.13, 0.58]

def modelo_ajuste_t0(t_janela, t0_livre, per, rp, a, inc):
    p = batman.TransitParams()
    p.t0, p.per, p.rp, p.a, p.inc = t0_livre, per, rp, a, inc
    p.ecc, p.w, p.u, p.limb_dark = 0.0, 90.0, LDC, "quadratic"
    return batman.TransitModel(p, t_janela).light_curve(p)

# ============================================================
# 2. FUNÇÃO MAPEADORA INTELIGENTE
# ============================================================
def mapear_transitos_hibrido(T0, T0_err, P, P_err, t_array, janela_dias=0.25, conhecidos={}):
    n_min = int(np.ceil((t_array.min() - T0) / P))
    n_max = int(np.floor((t_array.max() - T0) / P))
    
    eventos = [] 
    for n in range(n_min, n_max + 1):
        if n in conhecidos:
            tc_esperado = conhecidos[n]['tc']
            tc_err_esperado = conhecidos[n]['err']
            origem = "Literatura (Conhecido)"
        else:
            tc_esperado = T0 + n * P
            tc_err_esperado = np.sqrt(T0_err**2 + (n * P_err)**2)
            origem = "Teórico (Previsão Setor Novo)"
            
        pontos_na_janela = np.sum((t_array >= tc_esperado - janela_dias) & (t_array <= tc_esperado + janela_dias))
        
        if pontos_na_janela > 10:
            eventos.append({
                'epoch': n, 
                'tc_esperado': tc_esperado, 
                'tc_err': tc_err_esperado,
                'origem': origem
            })
    return eventos

# ============================================================
# 3. CLASSE PARA LOG_POSTERIOR
# ============================================================
class TransitFitter:
    def __init__(self, t_data, flux_data, flux_err, tc_prior, tc_prior_err):
        self.t_data = t_data
        self.flux_data = flux_data
        self.flux_err = flux_err
        self.tc_prior = tc_prior
        self.tc_prior_err = tc_prior_err
        
    def log_prior(self, theta):
        """Prior gaussiano no tempo central"""
        t0 = theta[0]
        # Rejeita valores muito distantes
        if np.abs(t0 - self.tc_prior) > 0.2:
            return -np.inf
        # Prior gaussiano
        return -0.5 * ((t0 - self.tc_prior) / self.tc_prior_err)**2
    
    def log_likelihood(self, theta):
        """Log-verossimilhança dos dados"""
        t0 = theta[0]
        try:
            modelo = modelo_ajuste_t0(self.t_data, t0, P_b, RP_b, A_b, INC_b)
            residuos = self.flux_data - modelo
            chi2 = np.sum((residuos / self.flux_err)**2)
            return -0.5 * chi2
        except:
            return -np.inf
    
    def log_posterior(self, theta):
        """Log-posteriori = log-prior + log-likelihood"""
        lp = self.log_prior(theta)
        if not np.isfinite(lp):
            return -np.inf
        return lp + self.log_likelihood(theta)

def run_mcmc_transit(t_data, flux_data, flux_err, tc_prior, tc_prior_err, 
                     n_walkers=32, n_steps=1000):
    """Executa MCMC para ajustar tempo central do trânsito"""
    
    fitter = TransitFitter(t_data, flux_data, flux_err, tc_prior, tc_prior_err)
    
    # Inicializa posições dos walkers - shape DEVE ser (n_walkers, ndim)
    p0 = (tc_prior + tc_prior_err * np.random.randn(n_walkers)).reshape(n_walkers, 1)
    
    # MCMC sem multiprocessing para evitar deadlock
    sampler = emcee.EnsembleSampler(n_walkers, 1, fitter.log_posterior)
    
    # Burn-in
    print("     → Burn-in (500 steps)...")
    state = sampler.run_mcmc(p0, 500, progress=False)
    sampler.reset()
    
    # Produção
    print("     → Production (1000 steps)...")
    sampler.run_mcmc(state, n_steps, progress=False)
    
    return sampler

# ============================================================
# 4. ROTINA PRINCIPAL DE AJUSTE COM MCMC
# ============================================================
%matplotlib qt

# Mapeia os trânsitos
eventos_b = mapear_transitos_hibrido(T0_b, T0_b_err, P_b, P_b_err, t, 0.25, transitos_conhecidos_b)

n_plots = len(eventos_b)
if n_plots > 0:
    print(f"Encontrados {n_plots} trânsitos!\n")
    
    resultados_mcmc = []
    
    fig, axes = plt.subplots(1, n_plots, figsize=(4.5 * n_plots, 4.0), sharey=True)
    if n_plots == 1: 
        axes = [axes]

    for i, ev in enumerate(eventos_b):
        epoch = ev['epoch']
        tc_esperado = ev['tc_esperado']
        tc_err_prior = ev['tc_err']
        origem = ev['origem']
        
        # Isola janela temporal
        janela = (t >= tc_esperado - 0.25) & (t <= tc_esperado + 0.25)
        t_f = t[janela]
        fluxo_f = residual_manchas[janela]
        
        # Incerteza: máximo de (ruído estimado, incerteza esperada)
        flux_err = np.ones_like(fluxo_f) * max(np.std(fluxo_f), 1e-4)
        
        print(f"Época {epoch:3d} ({origem[:15]})...")
        
        # MCMC
        sampler = run_mcmc_transit(
            t_f, fluxo_f, flux_err, 
            tc_esperado, tc_err_prior,
            n_walkers=32, n_steps=1000
        )
        
        # Extrai amostras (sem burn-in, pois foi descartado após reset)
        samples = sampler.get_chain(flat=True)
        tc_samples = samples[:, 0]
        
        # Estatísticas
        tc_medido = np.median(tc_samples)
        tc_err_medido = np.std(tc_samples)
        tc_16, tc_84 = np.percentile(tc_samples, [16, 84])
        
        # O-C
        tc_teorico = T0_b + epoch * P_b
        o_c_segundos = (tc_medido - tc_teorico) * 24 * 3600
        
        resultados_mcmc.append({
            'epoch': epoch,
            'tc_medido': tc_medido,
            'tc_err': tc_err_medido,
            'tc_16': tc_16,
            'tc_84': tc_84,
            'o_c_segundos': o_c_segundos,
            'origem': origem
        })
        
        print(f"  T_c: {tc_medido:.5f} ± {tc_err_medido:.5f} (prior: {tc_esperado:.5f})")
        print(f"  O-C: {o_c_segundos:+.1f} s")
        print(f"  68% CI: [{tc_16:.5f}, {tc_84:.5f}]\n")
        
        # Modelo ajustado
        fluxo_modelo = modelo_ajuste_t0(t_f, tc_medido, P_b, RP_b, A_b, INC_b)
        
        # Plot
        ax = axes[i]
        ax.plot(t_f, fluxo_f, 'k.', ms=2.5, alpha=0.4, label='Data')
        ax.plot(t_f, fluxo_modelo, 'r-', lw=2.5, label='MCMC fit', alpha=0.8)
        ax.axvline(tc_medido, color='red', ls='--', alpha=0.4, lw=1.5)
        ax.fill_betweenx([0.9975, 1.0025], tc_16, tc_84, alpha=0.25, color='red')
        
        cor = 'blue' if 'Novo' in origem else 'black'
        ax.set_title(f"Epoch {epoch}\nT_c={tc_medido:.4f}±{tc_err_medido:.5f}\nO-C={o_c_segundos:+.0f}s", 
                     fontsize=9, color=cor, fontweight='bold')
        ax.set_xlabel('Time [BTJD]', fontsize=9)
        ax.grid(alpha=0.25, linestyle=':')
        
        if i == 0:
            ax.set_ylabel('Flux', fontsize=9)
            ax.legend(loc='upper right', fontsize=8)

    plt.suptitle('AU Mic b - Transit Times (MCMC with Literature Priors)', 
                 fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # ============================================================
    # 5. TABELA DE EFEMÉRIDES
    # ============================================================
    print("\n" + "="*90)
    print("EPHEMERIDES - AU MIC B (MCMC Results)")
    print("="*90)
    print(f"{'Epoch':<8} {'T_c (BTJD)':<16} {'Uncertainty':<14} {'O-C (s)':<12} {'Origin':<30}")
    print("-"*90)
    
    for res in resultados_mcmc:
        print(f"{res['epoch']:<8} {res['tc_medido']:<16.6f} {res['tc_err']:<14.6f} "
              f"{res['o_c_segundos']:<12.1f} {res['origem']:<30}")
    
    print("="*90)
    
    # ============================================================
    # 6. DIAGRAMA O-C (OPCIONAL)
    # ============================================================
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    
    epochs = [r['epoch'] for r in resultados_mcmc]
    ocs = [r['o_c_segundos'] for r in resultados_mcmc]
    errs = [r['tc_err'] * 24 * 3600 for r in resultados_mcmc]  # converter para segundos
    
    colors = ['blue' if 'Novo' in r['origem'] else 'black' for r in resultados_mcmc]
    
    ax2.errorbar(epochs, ocs, yerr=errs, fmt='o', color='black', 
                 ecolor='gray', markersize=8, capsize=5, alpha=0.7, label='MCMC')
    ax2.axhline(0, color='red', ls='--', lw=2, alpha=0.5, label='Theory')
    ax2.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax2.set_ylabel('O-C (seconds)', fontsize=11, fontweight='bold')
    ax2.set_title('Transit Timing Variations - AU Mic b', fontsize=12, fontweight='bold')
    ax2.grid(alpha=0.3, linestyle=':')
    ax2.legend(fontsize=10)
    
    plt.tight_layout()
    plt.show()



PASSO 3: AJUSTE FOTOMÉTRICO COM MCMC (AU Mic b)


In [61]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import batman
import emcee

print("\n" + "="*60)
print("PASSO 3+4: AJUSTE FOTOMÉTRICO + PHASE-FOLD + MCMC (AU Mic b)")
print("="*60)

# ============================================================
# 1. DICIONÁRIO DE TRÂNSITOS CONHECIDOS (Setor 1 e 27)
# ============================================================
transitos_conhecidos_b = {
    0:  {'tc': 1330.39046, 'err': 0.00016},
    2:  {'tc': 1347.31646, 'err': 0.00016},
    84: {'tc': 2041.28238, 'err': 0.00026},
    85: {'tc': 2049.74538, 'err': 0.00026},
    86: {'tc': 2058.20838, 'err': 0.00026}
}

# Constantes do Planeta b
T0_b, T0_b_err = 1330.39051, 0.00015
P_b, P_b_err   = 8.463000, 0.000002
RP_b, A_b, INC_b = 0.0526, 19.1, 89.5
LDC = [0.13, 0.58]

def modelo_ajuste_t0(t_janela, t0_livre, per, rp, a, inc):
    p = batman.TransitParams()
    p.t0, p.per, p.rp, p.a, p.inc = t0_livre, per, rp, a, inc
    p.ecc, p.w, p.u, p.limb_dark = 0.0, 90.0, LDC, "quadratic"
    return batman.TransitModel(p, t_janela).light_curve(p)

# ============================================================
# 2. FUNÇÃO MAPEADORA (a mesma que já funciona pra você)
# ============================================================
def mapear_transitos_hibrido(T0, T0_err, P, P_err, t_array, janela_dias=0.25, conhecidos={}):
    n_min = int(np.ceil((t_array.min() - T0) / P))
    n_max = int(np.floor((t_array.max() - T0) / P))

    eventos = []
    for n in range(n_min, n_max + 1):
        if n in conhecidos:
            tc_esperado = conhecidos[n]['tc']
            tc_err_esperado = conhecidos[n]['err']
            origem = "Literatura (Conhecido)"
        else:
            tc_esperado = T0 + n * P
            tc_err_esperado = np.sqrt(T0_err**2 + (n * P_err)**2)
            origem = "Teórico (Previsão Setor Novo)"

        pontos_na_janela = np.sum((t_array >= tc_esperado - janela_dias) & (t_array <= tc_esperado + janela_dias))

        if pontos_na_janela > 10:
            eventos.append({
                'epoch': n,
                'tc_esperado': tc_esperado,
                'tc_err': tc_err_esperado,
                'origem': origem
            })
    return eventos

# Mapeia onde o planeta b passou nos dados
eventos_b = mapear_transitos_hibrido(T0_b, T0_b_err, P_b, P_b_err, t, 0.25, transitos_conhecidos_b)
print(f"Encontrados {len(eventos_b)} trânsitos de AU Mic b com dados!\n")

# ============================================================
# 3. EXTRAI OS DADOS DE CADA JANELA (t_f, fluxo_f) — sem ajuste ainda
# ============================================================
trechos = []
for ev in eventos_b:
    epoch = ev['epoch']
    tc_esperado = ev['tc_esperado']
    janela = (t >= tc_esperado - 0.25) & (t <= tc_esperado + 0.25)
    t_f, fluxo_f = t[janela], residual_manchas[janela]
    trechos.append({'epoch': epoch, 't': t_f, 'flux': fluxo_f, 'origem': ev['origem']})

if len(trechos) == 0:
    raise RuntimeError("Nenhum trânsito encontrado — confira t, residual_manchas, T0_b, P_b.")

t_all    = np.concatenate([tr['t']    for tr in trechos])
flux_all = np.concatenate([tr['flux'] for tr in trechos])
sigma    = np.std(flux_all - np.median(flux_all))
print(f"Total de pontos concatenados: {len(t_all)} | sigma estimado: {sigma:.2e}\n")

# ============================================================
# 4. MCMC — ajusta (T0, P) simultâneos = EFEMÉRIDES
# ============================================================
def modelo_TP(t_arr, t0, per):
    return modelo_ajuste_t0(t_arr, t0, per, RP_b, A_b, INC_b)

def log_prob(theta):
    T0, P = theta
    if not (T0_b - 0.02 < T0 < T0_b + 0.02): return -np.inf
    if not (P_b  - 0.01 < P  < P_b  + 0.01): return -np.inf
    lp  = -0.5 * ((T0 - T0_b) / T0_b_err)**2
    lp += -0.5 * ((P  - P_b ) / P_b_err )**2
    try:
        modelo = modelo_TP(t_all, T0, P)
        lnL = -0.5 * np.sum(((flux_all - modelo) / sigma)**2)
    except Exception:
        return -np.inf
    return lp + lnL

nwalkers, ndim, nsteps = 32, 2, 9000
p0 = np.array([T0_b, P_b]) + np.array([1e-5, 1e-6]) * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
print("Rodando MCMC (T0, P)...")
sampler.run_mcmc(p0, nsteps, progress=True)

samples = sampler.get_chain(discard=1000, thin=10, flat=True)
T0_fit, P_fit = np.median(samples, axis=0)
T0_err, P_err = np.std(samples, axis=0)

print("\n" + "="*60)
print("EFEMÉRIDES REFINADAS (MCMC)")
print("="*60)
print(f"T0 = {T0_fit:.6f} ± {T0_err:.6f}   (prior: {T0_b:.6f} ± {T0_b_err:.6f})")
print(f"P  = {P_fit:.6f} ± {P_err:.6f}   (prior: {P_b:.6f} ± {P_b_err:.6f})")
print("="*60)

# ============================================================
# 5. PHASE-FOLD: todas as curvas sobrepostas em uma janela só
# ============================================================
%matplotlib qt

def fase_dias(t_arr, T0, P):
    ph = ((t_arr - T0) / P + 0.5) % 1 - 0.5   # fase normalizada em [-0.5, 0.5]
    return ph * P                              # de volta para dias

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

cores = plt.cm.viridis(np.linspace(0, 0.9, len(trechos)))

for i, tr in enumerate(trechos):
    t_rel = fase_dias(tr['t'], T0_fit, P_fit)
    ax1.plot(t_rel, tr['flux'], 'o', ms=3, alpha=0.5, color=cores[i],
              label=f"Época {tr['epoch']} ({tr['origem'][:10]})")

t_mod = np.linspace(-0.25, 0.25, 500)
mod_fold = modelo_TP(T0_fit + t_mod, T0_fit, P_fit)
ax1.plot(t_mod, mod_fold, 'r-', lw=2.5, label='Modelo MCMC', zorder=100)

ax1.set_ylabel('Fluxo Residual')
ax1.set_title(f"AU Mic b — Trânsitos em Fase (todos sobrepostos)\n"
              f"T0={T0_fit:.6f}±{T0_err:.6f}   P={P_fit:.6f}±{P_err:.6f} d",
              fontweight='bold')
ax1.grid(alpha=0.3)
ax1.legend(fontsize=8, ncol=2, loc='lower right')

for i, tr in enumerate(trechos):
    t_rel = fase_dias(tr['t'], T0_fit, P_fit)
    modelo_pt = modelo_TP(T0_fit + t_rel, T0_fit, P_fit)
    res = tr['flux'] - modelo_pt
    ax2.plot(t_rel, res, 'o', ms=3, alpha=0.5, color=cores[i])

ax2.axhline(0, color='red', ls='--', lw=1.5, alpha=0.7)
ax2.set_xlabel('Tempo relativo a T0 (dias)')
ax2.set_ylabel('Resíduo')
ax2.set_xlim(-0.25, 0.25)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================
# 6. TABELA DE EFEMÉRIDES POR ÉPOCA (com T0, P refinados)
# ============================================================
print(f"\n{'Época':<8} {'T_c previsto (novo)':<22} {'Origem':<25}")
print("-"*60)
for tr in trechos:
    tc_novo = T0_fit + tr['epoch'] * P_fit
    print(f"{tr['epoch']:<8} {tc_novo:<22.6f} {tr['origem']:<25}")
print("="*60)

# ============================================================
# 7. CORNER PLOT (opcional)
# ============================================================
try:
    import corner
    corner.corner(samples, labels=[r"$T_0$", r"$P$"],
                  truths=[T0_b, P_b], show_titles=True)
    plt.show()
except ImportError:
    print("\n(dica: `pip install corner --break-system-packages` para ver o corner plot)")


PASSO 3+4: AJUSTE FOTOMÉTRICO + PHASE-FOLD + MCMC (AU Mic b)
Encontrados 0 trânsitos de AU Mic b com dados!



RuntimeError: Nenhum trânsito encontrado — confira t, residual_manchas, T0_b, P_b.

In [40]:
# ============================================================
# 5. PHASE-FOLD: todas as curvas sobrepostas em uma janela só
#    + PAINEL EXTRA: todos os trânsitos individuais empilhados
# ============================================================
%matplotlib qt

def fase_dias(t_arr, T0, P):
    ph = ((t_arr - T0) / P + 0.5) % 1 - 0.5   # fase normalizada em [-0.5, 0.5]
    return ph * P                              # de volta para dias

fig = plt.figure(figsize=(11, 12))
gs = fig.add_gridspec(3, 1, height_ratios=[3, 1, 3.5], hspace=0.4)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax3 = fig.add_subplot(gs[2])

cores = plt.cm.viridis(np.linspace(0, 0.9, len(trechos)))

# --- Painel 1: fold sobreposto ---
for i, tr in enumerate(trechos):
    t_rel = fase_dias(tr['t'], T0_fit, P_fit)
    ax1.plot(t_rel, tr['flux'], 'o', ms=3, alpha=0.5, color=cores[i],
              label=f"Época {tr['epoch']} ({tr['origem'][:10]})")

t_mod = np.linspace(-0.25, 0.25, 500)
mod_fold = modelo_TP(T0_fit + t_mod, T0_fit, P_fit)
ax1.plot(t_mod, mod_fold, 'r-', lw=2.5, label='Modelo MCMC', zorder=100)

ax1.set_ylabel('Fluxo Residual')
ax1.set_title(f"AU Mic b — Trânsitos em Fase (todos sobrepostos)\n"
              f"T0={T0_fit:.6f}±{T0_err:.6f}   P={P_fit:.6f}±{P_err:.6f} d",
              fontweight='bold')
ax1.grid(alpha=0.3)
ax1.legend(fontsize=8, ncol=2, loc='lower right')

# --- Painel 2: resíduos do fold ---
for i, tr in enumerate(trechos):
    t_rel = fase_dias(tr['t'], T0_fit, P_fit)
    modelo_pt = modelo_TP(T0_fit + t_rel, T0_fit, P_fit)
    res = tr['flux'] - modelo_pt
    ax2.plot(t_rel, res, 'o', ms=3, alpha=0.5, color=cores[i])

ax2.axhline(0, color='red', ls='--', lw=1.5, alpha=0.7)
ax2.set_xlabel('Tempo relativo a T0 (dias)')
ax2.set_ylabel('Resíduo')
ax2.set_xlim(-0.25, 0.25)
ax2.grid(alpha=0.3)

# --- Painel 3: TODOS os trânsitos individuais, empilhados ---
n_trans = len(trechos)
offset_step = 0.010  # espaçamento vertical entre curvas

for i, tr in enumerate(trechos):
    t_rel_indiv = tr['t'] - (T0_fit + tr['epoch'] * P_fit)  # tempo relativo ao próprio T_c
    flux_offset = tr['flux'] + i * offset_step

    modelo_indiv = modelo_TP(tr['t'], T0_fit, P_fit)  # periódico: acerta cada época automaticamente

    ax3.plot(t_rel_indiv, flux_offset, 'o', ms=3, alpha=0.5, color=cores[i])
    ax3.plot(t_rel_indiv, modelo_indiv + i * offset_step, '-', lw=1.8, color='crimson', alpha=0.9)
    ax3.text(0.26, 1 + i * offset_step, f"Época {tr['epoch']}", fontsize=8,
             va='center', color=cores[i])

ax3.set_xlabel('Tempo relativo ao T_c da própria época (dias)')
ax3.set_ylabel('Fluxo (com offset)')
ax3.set_title(f'Todos os {n_trans} Trânsitos Individuais (empilhados, sem dobrar)', fontweight='bold')
ax3.set_xlim(-0.25, 0.32)
ax3.set_ylim(1 - 3*offset_step, 1 + n_trans * offset_step + 3*offset_step)  # cabe todos, com folga
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [8]:
# ============================================================
# 2.5. CHECAGEM VISUAL — trânsitos em fase ANTES do MCMC
#      (usa os priors T0_pri, P_pri para conferir a extração)
# ============================================================
%matplotlib qt

def fase_dias(t_arr, T0, P):
    ph = ((t_arr - T0) / P + 0.5) % 1 - 0.5   # fase em [-0.5, +0.5]
    return ph * P                              # em dias

fig_check, (axc1, axc2) = plt.subplots(2, 1, figsize=(9, 8),
                                        gridspec_kw={'height_ratios': [2, 3]})

cores = plt.cm.viridis(np.linspace(0, 0.9, len(trechos)))

# --- Painel de cima: todos sobrepostos em fase (com o PRIOR) ---
for i, tr in enumerate(trechos):
    t_rel = fase_dias(tr['t'], T0_pri, P_pri) * 24   # em horas
    axc1.plot(t_rel, tr['flux'], 'o', ms=3, alpha=0.5, color=cores[i],
              label=f"Época {tr['epoch']} ({tr['origem'][:10]})")

# modelo do prior por cima
t_mod = np.linspace(-0.25, 0.25, 400)
mod_prior = modelo_TP(T0_pri + t_mod, T0_pri, P_pri)
axc1.plot(t_mod * 24, mod_prior, 'r-', lw=2.5, label='Modelo (prior)', zorder=100)
axc1.axvline(0, color='gray', ls=':', lw=1)

axc1.set_xlabel('Tempo relativo ao $T_c$ prior (horas)')
axc1.set_ylabel('Fluxo')
axc1.set_title(f'Trânsitos em FASE com o PRIOR (antes do MCMC)\n'
               f'T0_pri={T0_pri:.5f}   P_pri={P_pri:.6f} d', fontweight='bold', fontsize=10)
axc1.set_xlim(-6, 6)
axc1.grid(alpha=0.3)
axc1.legend(fontsize=8, ncol=2)

# --- Painel de baixo: trânsitos empilhados (offset) ---
offset_step = 0.006
for i, tr in enumerate(trechos):
    off = i * offset_step
    t_rel = fase_dias(tr['t'], T0_pri, P_pri) * 24
    axc2.plot(t_rel, tr['flux'] + off, 'o', ms=3, alpha=0.5, color=cores[i])
    axc2.plot(t_mod * 24, mod_prior + off, 'r-', lw=1.5, alpha=0.8)
    axc2.text(6.3, 1 + off, f"Ép. {tr['epoch']} ({tr['origem'][:6]})",
              fontsize=8, va='center', color=cores[i])

axc2.axvline(0, color='gray', ls=':', lw=1)
axc2.set_xlabel('Tempo relativo ao $T_c$ prior (horas)')
axc2.set_ylabel('Fluxo (empilhado)')
axc2.set_title('Trânsitos empilhados (checagem individual)', fontweight='bold', fontsize=10)
axc2.set_xlim(-6, 9)
axc2.set_ylim(1 - 3*offset_step, 1 + len(trechos)*offset_step + 3*offset_step)
axc2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n>>> Confira se os trânsitos estão centrados e alinhados.")
print(">>> Se estiverem tortos/deslocados, ajuste T0_pri/P_pri antes de rodar o MCMC.\n")


>>> Confira se os trânsitos estão centrados e alinhados.
>>> Se estiverem tortos/deslocados, ajuste T0_pri/P_pri antes de rodar o MCMC.



In [4]:
import numpy as np
import matplotlib.pyplot as plt
import batman
import emcee

print("\n" + "="*60)
print("AJUSTE DE EFEMÉRIDE (T0, P) + TRÂNSITOS SEPARADOS — AU Mic b")
print("="*60)

# ============================================================
# 1. PARÂMETROS
# ============================================================
transitos_conhecidos_b = {
    0:  {'tc': 1330.39046, 'err': 0.00016},
    2:  {'tc': 1347.31646, 'err': 0.00016},
    84: {'tc': 2041.28238, 'err': 0.00026},
    85: {'tc': 2049.74538, 'err': 0.00026},
    86: {'tc': 2058.20838, 'err': 0.00026}
}

T0_pri, T0_pri_err = 1330.38957, 0.00015
P_pri,  P_pri_err  = 8.463000,   0.000002
RP_b, A_b, INC_b = 0.0496, 19.1, 89.5   # FIXOS
LDC = [0.13, 0.58]

def modelo_TP(t_arr, T0, P):
    p = batman.TransitParams()
    p.t0, p.per, p.rp, p.a, p.inc = T0, P, RP_b, A_b, INC_b
    p.ecc, p.w, p.u, p.limb_dark = 0.0, 90.0, LDC, "quadratic"
    return batman.TransitModel(p, t_arr).light_curve(p)

# ============================================================
# 2. MAPEAMENTO E EXTRAÇÃO
# ============================================================
def mapear_transitos_hibrido(T0, T0_err, P, P_err, t_array, janela_dias=0.25, conhecidos={}):
    n_min = int(np.ceil((t_array.min() - T0) / P))
    n_max = int(np.floor((t_array.max() - T0) / P))
    eventos = []
    for n in range(n_min, n_max + 1):
        if n in conhecidos:
            tc_esperado = conhecidos[n]['tc']
            origem = "Literatura"
        else:
            tc_esperado = T0 + n * P
            origem = "Teórico"
        pontos = np.sum((t_array >= tc_esperado - janela_dias) & (t_array <= tc_esperado + janela_dias))
        if pontos > 10:
            eventos.append({'epoch': n, 'tc_esperado': tc_esperado, 'origem': origem})
    return eventos

eventos_b = mapear_transitos_hibrido(T0_pri, T0_pri_err, P_pri, P_pri_err,
                                      t, 0.25, transitos_conhecidos_b)
print(f"Encontrados {len(eventos_b)} trânsitos\n")

trechos = []
for ev in eventos_b:
    janela = (t >= ev['tc_esperado'] - 0.25) & (t <= ev['tc_esperado'] + 0.25)
    trechos.append({'epoch': ev['epoch'], 't': t[janela],
                    'flux': residual_manchas[janela], 'origem': ev['origem']})

if len(trechos) == 0:
    raise RuntimeError("Nenhum trânsito encontrado.")

t_all    = np.concatenate([tr['t']    for tr in trechos])
flux_all = np.concatenate([tr['flux'] for tr in trechos])
sigma    = np.std(flux_all - np.median(flux_all))
print(f"Total: {len(t_all)} pontos em {len(trechos)} trânsitos\n")

# ============================================================
# 3. MCMC — SÓ T0 E P
# ============================================================
def log_prob(theta):
    T0, P = theta
    if not (T0_pri - 0.02 < T0 < T0_pri + 0.02): return -np.inf
    if not (P_pri  - 0.01 < P  < P_pri  + 0.01): return -np.inf
    lp  = -0.5 * ((T0 - T0_pri) / T0_pri_err)**2
    lp += -0.5 * ((P  - P_pri ) / P_pri_err )**2
    try:
        modelo = modelo_TP(t_all, T0, P)
        lnL = -0.5 * np.sum(((flux_all - modelo) / sigma)**2)
    except Exception:
        return -np.inf
    return lp + lnL

nwalkers, ndim, nsteps = 100, 2, 30000
p0 = np.array([T0_pri, P_pri]) + np.array([1e-5, 1e-6]) * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
print(f"MCMC: {nwalkers} walkers × {nsteps} steps")
print("Rodando...")
sampler.run_mcmc(p0, nsteps, progress=True)

try:
    tau = sampler.get_autocorr_time(quiet=True)
    discard = int(2 * np.max(tau))
    thin    = max(int(0.5 * np.min(tau)), 1)
    print(f"\nAutocorr: T0={tau[0]:.0f}  P={tau[1]:.0f}")
except Exception:
    discard, thin = 2000, 15

samples = sampler.get_chain(discard=discard, thin=thin, flat=True)
T0_fit, P_fit = np.median(samples, axis=0)
T0_err, P_err = np.std(samples, axis=0)

print("\n" + "="*60)
print("EFEMÉRIDE REFINADA (MCMC)")
print("="*60)
print(f"T0 = {T0_fit:.6f} ± {T0_err:.6f}   (prior: {T0_pri:.6f} ± {T0_pri_err:.6f})")
print(f"P  = {P_fit:.6f} ± {P_err:.6f}   (prior: {P_pri:.6f} ± {P_pri_err:.6f})")
print("="*60)

# ============================================================
# 4. TRÂNSITOS SEPARADOS (grid) COM EFEMÉRIDE NOVA
# ============================================================
%matplotlib qt

n = len(trechos)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4.5), sharey=True)
if n == 1: axes = [axes]

print(f"\n{'Época':<8} {'T_c novo (efeméride)':<24} {'Origem':<12}")
print("-"*50)

for i, tr in enumerate(trechos):
    ax = axes[i]
    epoch = tr['epoch']

    # T_c pela EFEMÉRIDE NOVA
    tc_novo = T0_fit + epoch * P_fit

    # Modelo com a efeméride nova (batman é periódico, acerta a época sozinho)
    t_denso = np.linspace(tr['t'].min(), tr['t'].max(), 400)
    modelo_denso = modelo_TP(t_denso, T0_fit, P_fit)

    # Plot
    ax.plot(tr['t'], tr['flux'], 'k.', ms=3, alpha=0.4, label='Dados')
    ax.plot(t_denso, modelo_denso, 'r-', lw=2.5, label='Efeméride nova')
    ax.axvline(tc_novo, color='red', ls='--', alpha=0.6)

    cor = 'blue' if 'Teór' in tr['origem'] else 'black'
    ax.set_title(f"Época {epoch}\n$T_c$ = {tc_novo:.5f}", fontsize=10, color=cor)
    ax.set_xlabel('Tempo [BTJD]')
    ax.grid(alpha=0.3)
    if i == 0:
        ax.set_ylabel('Fluxo Residual')
        ax.legend(fontsize=8)

    print(f"{epoch:<8} {tc_novo:<24.6f} {tr['origem']:<12}")

print("="*50)

plt.suptitle(f'AU Mic b — Trânsitos individuais com efeméride nova\n'
             f'T0={T0_fit:.6f}±{T0_err:.6f}   P={P_fit:.6f}±{P_err:.6f} d',
             fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

# ============================================================
# 5. CORNER PLOT (opcional)
# ============================================================
try:
    import corner
    corner.corner(samples, labels=[r"$T_0$", r"$P$"],
                  truths=[T0_pri, P_pri], quantiles=[0.16, 0.5, 0.84],
                  show_titles=True, title_fmt='.6f')
    plt.show()
except ImportError:
    print("\n(pip install corner --break-system-packages para o corner plot)")


AJUSTE DE EFEMÉRIDE (T0, P) + TRÂNSITOS SEPARADOS — AU Mic b
Encontrados 7 trânsitos

Total: 2458 pontos em 7 trânsitos

MCMC: 100 walkers × 30000 steps
Rodando...


100%|██████████| 30000/30000 [41:41<00:00, 11.99it/s] 



Autocorr: T0=32  P=32

EFEMÉRIDE REFINADA (MCMC)
T0 = 1330.389569 ± 0.000146   (prior: 1330.389570 ± 0.000150)
P  = 8.463001 ± 0.000002   (prior: 8.463000 ± 0.000002)

Época    T_c novo (efeméride)     Origem      
--------------------------------------------------
0        1330.389569              Literatura  
2        1347.315571              Literatura  
84       2041.281658              Literatura  
85       2049.744659              Literatura  
86       2058.207660              Literatura  
302      3886.215889              Teórico     
304      3903.141891              Teórico     


In [5]:
fig, axes = plt.subplots(2, figsize=(10, 6), sharex=True)
samples_raw = sampler.get_chain()
labels = [r"$T_0$", r"$P$"]

for i in range(2):
    ax = axes[i]
    ax.plot(samples_raw[:, :, i], "k", alpha=0.3)
    ax.set_ylabel(labels[i])
    ax.axvline(discard, color='r', linestyle='--', label='Burn-in')

axes[1].set_xlabel("Passos (Steps)")
axes[0].set_title("Trace Plot - Verificação Visual de Convergência")
axes[0].legend()
plt.show()

In [6]:
%matplotlib qt

# ============================================================
# GRÁFICO: 7 trânsitos empilhados com a efeméride nova
# ============================================================
fig, ax = plt.subplots(figsize=(9, 11))

offset_step = 0.006   # espaçamento vertical entre trânsitos
cores = plt.cm.viridis(np.linspace(0, 0.9, len(trechos)))

for i, tr in enumerate(trechos):
    epoch = tr['epoch']
    off = i * offset_step

    # T_c da efeméride nova
    tc_novo = T0_fit + epoch * P_fit

    # Tempo relativo ao próprio T_c (centra cada trânsito em zero)
    t_rel = tr['t'] - tc_novo

    # Modelo denso da efeméride nova, também centrado
    t_denso = np.linspace(tr['t'].min(), tr['t'].max(), 400)
    modelo_denso = modelo_TP(t_denso, T0_fit, P_fit)
    t_denso_rel = t_denso - tc_novo

    # Dados + modelo com offset
    ax.plot(t_rel * 24, tr['flux'] + off, 'o', ms=3, alpha=0.4, color=cores[i])
    ax.plot(t_denso_rel * 24, modelo_denso + off, '-', lw=2, color='crimson', zorder=50)

    # Linha do T_c (centro do trânsito = 0)
    ax.axvline(0, color='gray', ls=':', lw=0.8, alpha=0.5)

    # Rótulo de cada época
    cor_txt = 'blue' if 'Teór' in tr['origem'] else 'black'
    ax.text(6.3, 1 + off, f"Época {epoch}\n{tr['origem'][:10]}",
            fontsize=8, va='center', color=cor_txt)
    ax.text(-6.3, 1 + off, f"$T_c$={tc_novo:.4f}",
            fontsize=7, va='center', ha='left', color=cor_txt, alpha=0.8)

ax.set_xlabel('Tempo relativo ao $T_c$ (horas)', fontweight='bold')
ax.set_ylabel('Fluxo Normalizado (com offset)', fontweight='bold')
ax.set_title(f'AU Mic b — {len(trechos)} Trânsitos com Efeméride Nova\n'
             f'T0 = {T0_fit:.6f} ± {T0_err:.6f}   |   '
             f'P = {P_fit:.6f} ± {P_err:.6f} d',
             fontweight='bold', fontsize=11)
ax.set_xlim(-8, 9)
ax.set_ylim(1 - 3*offset_step, 1 + len(trechos)*offset_step + 3*offset_step)
ax.grid(alpha=0.2, axis='x')

plt.tight_layout()
plt.show()

In [7]:
# ============================================================
# COMPARAÇÃO EFEMÉRIDE OLD VS NOVA - AU Mic b
# COM ERROS DE T0
# ============================================================

%matplotlib qt

import numpy as np
import matplotlib.pyplot as plt


print("\n" + "="*70)
print("COMPARAÇÃO DAS EFEMÉRIDES AU Mic b")
print("="*70)


# ============================================================
# EFEMÉRIDE OLD
# ============================================================

T0_old = 1330.39051
erro_T0_old = 0.00016

# colocar aqui o período antigo quando disponível
P_old = 8.4631516


# ============================================================
# EFEMÉRIDE NOVA
# ============================================================

T0_new = 1330.389569
erro_T0_new = 0.000146

P_new = 8.4631516


print("\nEFEMÉRIDE OLD")
print("-"*40)
print(f"T0 = {T0_old:.6f} ± {erro_T0_old:.6f} BTJD")
print(f"P  = {P_old:.7f} dias")
print(f"Erro T0 = {erro_T0_old*24*60:.2f} minutos")


print("\nEFEMÉRIDE NOVA")
print("-"*40)
print(f"T0 = {T0_new:.6f} ± {erro_T0_new:.6f} BTJD")
print(f"P  = {P_new:.7f} dias")
print(f"Erro T0 = {erro_T0_new*24*60:.2f} minutos")


# diferença entre efemérides

delta = (T0_old - T0_new)*24*60

print("\nDiferença entre T0:")
print(f"{delta:.2f} minutos")


# ============================================================
# INTERVALO DOS DADOS
# ============================================================

tempo_min = np.min([np.min(t) for t in t_todos])
tempo_max = np.max([np.max(t) for t in t_todos])


# ============================================================
# CALCULA TRÂNSITOS OLD
# ============================================================

nmin_old = int(np.floor((tempo_min-T0_old)/P_old))
nmax_old = int(np.ceil((tempo_max-T0_old)/P_old))

ep_old = np.arange(
    nmin_old,
    nmax_old+1
)

trans_old = T0_old + ep_old*P_old



# ============================================================
# CALCULA TRÂNSITOS NOVA
# ============================================================

nmin_new = int(np.floor((tempo_min-T0_new)/P_new))
nmax_new = int(np.ceil((tempo_max-T0_new)/P_new))

ep_new = np.arange(
    nmin_new,
    nmax_new+1
)

trans_new = T0_new + ep_new*P_new



# ============================================================
# MOSTRA LISTA DOS TRÂNSITOS
# ============================================================

print("\n")
print("="*70)
print("TRÂNSITOS PREVISTOS")
print("="*70)


print("\nOLD")
for e,t in zip(ep_old,trans_old):
    print(f"Época {e:4d}  {t:.6f} BTJD")


print("\nNOVA")
for e,t in zip(ep_new,trans_new):
    print(f"Época {e:4d}  {t:.6f} BTJD")



# ============================================================
# NOMES DOS SETORES
# ============================================================

setor_nomes = [
    "Setor 1 (2018)",
    "Setor 27 (2020)",
    "Setor 95 (2025)"
]



# ============================================================
# FIGURA
# ============================================================

fig, axes = plt.subplots(
    3,
    1,
    figsize=(14,10)
)



for i,(t_sec,f_sec) in enumerate(zip(t_todos,f_todos)):

    ax = axes[i]


    # dados

    ax.plot(
        t_sec,
        f_sec,
        'k.',
        markersize=2,
        alpha=0.5,
        label="Dados TESS"
    )



    # ========================================================
    # OLD
    # ========================================================

    first_old=True

    for e,tr in zip(ep_old,trans_old):

        if t_sec[0] <= tr <= t_sec[-1]:


            ax.axvline(
                tr,
                color='blue',
                linestyle='--',
                linewidth=1.8,
                label="Efeméride OLD" if first_old else None
            )


            ax.axvspan(
                tr-erro_T0_old,
                tr+erro_T0_old,
                color='blue',
                alpha=0.15,
                label="Erro OLD (1σ)" if first_old else None
            )


            ax.text(
                tr,
                np.nanmax(f_sec),
                f"E{e}",
                color="blue",
                rotation=90,
                fontsize=8
            )


            first_old=False



    # ========================================================
    # NOVA
    # ========================================================

    first_new=True

    for e,tr in zip(ep_new,trans_new):

        if t_sec[0] <= tr <= t_sec[-1]:


            ax.axvline(
                tr,
                color='red',
                linestyle='-',
                linewidth=1.8,
                label="Efeméride NOVA" if first_new else None
            )


            ax.axvspan(
                tr-erro_T0_new,
                tr+erro_T0_new,
                color='red',
                alpha=0.15,
                label="Erro NOVA (1σ)" if first_new else None
            )


            ax.text(
                tr,
                np.nanmin(f_sec),
                f"E{e}",
                color="red",
                rotation=90,
                fontsize=8
            )


            first_new=False



    # ========================================================
    # FORMATAÇÃO
    # ========================================================

    ax.set_title(
        setor_nomes[i],
        fontsize=13,
        fontweight='bold'
    )


    ax.set_ylabel(
        "Fluxo normalizado"
    )


    ax.grid(
        alpha=0.3,
        linestyle=":"
    )


    ax.legend(
        fontsize=9,
        loc="best"
    )



axes[-1].set_xlabel(
    "Tempo [BTJD]",
    fontsize=12
)



plt.suptitle(
    "AU Mic b - Comparação Efeméride OLD vs NOVA\n"
    "Azul = OLD | Vermelho = NOVA",
    fontsize=15,
    fontweight="bold"
)


plt.tight_layout()

plt.show()



print("\n✓ Comparação finalizada!")


COMPARAÇÃO DAS EFEMÉRIDES AU Mic b

EFEMÉRIDE OLD
----------------------------------------
T0 = 1330.390510 ± 0.000160 BTJD
P  = 8.4631516 dias
Erro T0 = 0.23 minutos

EFEMÉRIDE NOVA
----------------------------------------
T0 = 1330.389569 ± 0.000146 BTJD
P  = 8.4631516 dias
Erro T0 = 0.21 minutos

Diferença entre T0:
1.36 minutos


TRÂNSITOS PREVISTOS

OLD
Época   -1  1321.927358 BTJD
Época    0  1330.390510 BTJD
Época    1  1338.853662 BTJD
Época    2  1347.316813 BTJD
Época    3  1355.779965 BTJD
Época    4  1364.243116 BTJD
Época    5  1372.706268 BTJD
Época    6  1381.169420 BTJD
Época    7  1389.632571 BTJD
Época    8  1398.095723 BTJD
Época    9  1406.558874 BTJD
Época   10  1415.022026 BTJD
Época   11  1423.485178 BTJD
Época   12  1431.948329 BTJD
Época   13  1440.411481 BTJD
Época   14  1448.874632 BTJD
Época   15  1457.337784 BTJD
Época   16  1465.800936 BTJD
Época   17  1474.264087 BTJD
Época   18  1482.727239 BTJD
Época   19  1491.190390 BTJD
Época   20  1499.653542 BTJD
